# Steam 리뷰 LLM 감성·이슈 분석 실행 파일



## 역할
1. 전처리된 Steam 리뷰/게임 메타/리뷰 히스토그램 데이터를 불러온다.
2. 분석 조건에 맞게 리뷰를 필터링하고 게임별로 샘플링한다.
3. Gemini LLM으로 리뷰 감성, 핵심 이슈, 세부 이슈, 개선 방향을 분석한다.
4. 보고서 노트북이 바로 읽을 수 있는 핵심 CSV/JSON 산출물만 저장한다.


## 최종 산출물

| 파일 | 역할 |
|---|---|
| `llm_review_analysis_result.json` | 리뷰별 LLM 분석 결과 JSON |
| `llm_review_analysis_result.csv` | 리뷰별 LLM 분석 결과 CSV |
| `llm_review_analysis_checkpoint.json` | LLM 분석 중간 저장 파일 |
| `llm_game_summary.csv` | 게임 단위 요약표 |
| `llm_issue_tags_flat.csv` | 리뷰별 세부 이슈 태그를 행 단위로 펼친 파일 |
| `llm_issue_priority_summary.csv` | 이슈별 긍정/부정 방향성 요약 파일 |

> 보고서용 표, 그래프, 해석 문장은 `report` 코드에서 담당


## 분석흐름
0. 사용 설명서
1. 환경 설정
2. 분석 조건 설정
3. 공통 헬퍼 함수
4. 데이터 로드
5. 데이터 기본 점검
6. 리뷰 데이터 전처리
7. 게임 메타 데이터 전처리
8. 리뷰 히스토그램 데이터 전처리
9. 리뷰 + 게임 메타 + 히스토그램 요약 결합
10. 분석 조건 필터링
11. 게임당 리뷰 샘플링
12. LLM 출력 스키마 정의
13. Agent 생성
14. 프롬프트 생성
15. 비동기 LLM 분석 실행
16. 결과 후처리 및 저장
17. 게임 단위 요약표 생성
18. 이슈 태그 펼치기
19. 최종 사용 메모

## 본 분석 흐름

1. 입력 파일 로드
2. 리뷰/게임 메타/히스토그램 전처리
3. 분석 조건 필터링
4. 게임별 리뷰 샘플링
5. LLM 분석 실행 또는 기존 결과 불러오기
6. 성공 결과와 실패 로그 분리 저장
7. 보고서용 요약 CSV 생성


# 0. 환경설정

아래 패키지가 설치되어 있지 않다면 터미널에서 먼저 설치
```bash
uv pip install "pydantic-ai-slim[google]" tqdm pandas python-dotenv
```

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import ast
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional
from datetime import datetime

# ============================================================
# 데이터 분석용 라이브러리
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ============================================================
# 환경변수 / 진행률 / LLM 출력 스키마 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings



# ============================================================
# 한글 폰트 설정
# ============================================================
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:  # Linux
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

c:\Users\joon5\Documents\github\steam-indie-game-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 경로 설정

In [2]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정합니다.
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# 소스 파일이 들어있는 폴더
# 리뷰 파일은 data/preprocessed 폴더 기준으로 불러온다.
DATA_DIR = ROOT / "data" / "preprocessed"

# 실제 분석에 사용할 파일명
REVIEW_FILE = "steam_indie_reviews.csv"                  # 전처리된 리뷰 본문 파일
GAME_META_FILE = "steam_indie_games.csv"                 # 게임명/장르/태그/출시일 등 메타데이터
REVIEW_HISTOGRAM_FILE = "steam_indie_review_histogram.csv" # 날짜별 리뷰 증감 히스토그램


# 전체 경로 생성
INPUT_REVIEW_PATH = DATA_DIR / REVIEW_FILE
GAME_META_PATH = DATA_DIR / GAME_META_FILE
REVIEW_HISTOGRAM_PATH = DATA_DIR / REVIEW_HISTOGRAM_FILE

print("호출 폴더")
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("리뷰 파일:", INPUT_REVIEW_PATH)
print("게임 메타 파일:", GAME_META_PATH)
print("리뷰 히스토그램 파일:", REVIEW_HISTOGRAM_PATH)

호출 폴더
ROOT: C:\Users\joon5\Documents\github\steam-indie-game-analysis
DATA_DIR: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\preprocessed
리뷰 파일: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\preprocessed\steam_indie_reviews.csv
게임 메타 파일: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\preprocessed\steam_indie_games.csv
리뷰 히스토그램 파일: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\preprocessed\steam_indie_review_histogram.csv


## Gemini API / 저장 경로 설정

`.env` 파일에서 Gemini API Key를 불러오고, 결과 저장 파일 경로를 만든다.

In [3]:
# ============================================================
# Gemini API / 결과 저장 설정
# ============================================================
# .env 파일에 저장된 환경변수를 현재 Python 환경으로 불러온다.
#
# .env 파일 예시:
# GEMINI_API_KEY=본인_API_KEY
# GEMINI_MODEL=gemini-2.5-flash
# ------------------------------------------------------------

load_dotenv()
load_dotenv(ROOT / ".env")

# .env에서 API Key와 모델명을 읽어온다.
api_key = os.getenv("GEMINI_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# Google Gemini 모델을 지정할 때 쓰는 모델 ID 형식
model_id = f"google-gla:{gemini_model}"

print("API 키 설정 확인:", "O" if api_key else "X")
print("사용 모델:", model_id)

# ============================================================
# 본 분석 실행 이름
# ============================================================
RUN_NAME = "main_action_d0_d30"

# ============================================================
# 결과 저장 폴더
# ============================================================
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 저장 산출물
# ============================================================

# LLM 분석 중간 저장 파일
CHECKPOINT_PATH = OUTPUT_DIR / "llm_review_analysis_checkpoint.json"

# 리뷰 단위 LLM 분석 결과
RESULT_JSON_PATH = OUTPUT_DIR / "llm_review_analysis_result.json"
RESULT_CSV_PATH = OUTPUT_DIR / "llm_review_analysis_result.csv"

# 보고서용 요약 산출물
GAME_SUMMARY_PATH = OUTPUT_DIR / "llm_game_summary.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "llm_issue_tags_flat.csv"
ISSUE_PRIORITY_PATH = OUTPUT_DIR / "llm_issue_priority_summary.csv"

print("본 분석 실행명:", RUN_NAME)
print("결과 저장 폴더:", OUTPUT_DIR)
print("저장 파일")
print("-", RESULT_CSV_PATH.name)
print("-", RESULT_JSON_PATH.name)
print("-", CHECKPOINT_PATH.name)
print("-", GAME_SUMMARY_PATH.name)
print("-", ISSUE_TAG_FLAT_PATH.name)
print("-", ISSUE_PRIORITY_PATH.name)

API 키 설정 확인: O
사용 모델: google-gla:gemini-2.5-flash
본 분석 실행명: main_action_d0_d30
결과 저장 폴더: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30
저장 파일
- llm_review_analysis_result.csv
- llm_review_analysis_result.json
- llm_review_analysis_checkpoint.json
- llm_game_summary.csv
- llm_issue_tags_flat.csv
- llm_issue_priority_summary.csv


# 1. 분석 조건 설정
이 부분만 수정하면 분석 대상을 바꿀 수 있다.




| 구분 | 의미 |
| --- | --- |
| 1-1. 실행 여부 / 테스트 여부 | LLM을 실제로 실행할지, 중간 점검을 출력할지, 기존 결과를 다시 사용할지 정하는 설정 |
| 1-2. 샘플링 옵션 | 게임별로 리뷰를 몇 개 뽑고, 어떤 방식으로 리뷰를 선택할지 정하는 설정 |
| 1-3. LLM 호출 옵션 | LLM 요청을 한 번에 몇 개씩 보내고, 실패 시 재시도나 요청 간 대기 시간을 어떻게 둘지 정하는 설정 |
| 1-5. 분석 대상 필터 설정 | 어떤 게임, 장르, 태그, 언어, 긍정/부정 리뷰를 분석 대상으로 삼을지 정하는 설정 |

아래 설명 셀은 각 설정값의 의미를 정리한 것이고,\
실제 실행값은 바로 아래의 짧은 코드 셀에서만 수정

## 1-1. 실행 여부 / 테스트 여부

- RUN_LLM = False
    - True:\
    실제 Gemini API를 호출해서 리뷰 감성/이슈 분석을 수행한다.
    - False:\
    Gemini API를 호출하지 않는다.\
    이미 저장된 RESULT_JSON 또는 CHECKPOINT가 있다면 그것을 불러와 후속 요약표만 만든다.\
    처음 노트북 구조를 확인할 때는 False로 두는 것을 권장

- RESET_CHECKPOINT = True
    - True:\
    기존 checkpoint 파일을 삭제하고 처음부터 다시 분석한다.\
    프롬프트, 출력 스키마, 분석 기준을 크게 바꾼 뒤에는 True로 한 번 실행하는 것이 좋다.

    - False:\
    기존 checkpoint를 유지한다.\
    이미 분석한 리뷰는 다시 호출하지 않는다.

- RUN_CHECK_CELLS = True
    - True:\
    중간 점검표와 분포표를 출력한다.\
    데이터가 제대로 들어왔는지 확인할 때 사용한다.
    - False:\
    점검 출력 없이 핵심 분석만 실행한다.

In [4]:
RUN_LLM = True
RESET_CHECKPOINT = True
RUN_CHECK_CELLS = True

## 1-2. 샘플링 옵션

- TEST_N = None
    - TEST_N = 30\
    최종 샘플 중 30개만 LLM 분석한다.\
    코드가 제대로 돌아가는지 확인할 때 사용

    - TEST_N = None\
    최종 샘플링된 리뷰 전체를 LLM 분석한다.\
    본격 실행 전에 필터와 샘플 수를 꼭 확인해야 한다.

- RANDOM_STATE = 42
    - 무작위 샘플링 결과를 재현하기 위한 숫자
    - 같은 숫자를 쓰면 같은 조건에서 같은 샘플이 뽑힌다.


- REVIEWS_PER_GAME = 4
    - 게임 1개당 최대 몇 개 리뷰를 LLM 분석에 사용할지 설정한다.
    - 특정 게임의 리뷰가 너무 많으면 그 게임만 결과를 지배할 수 있다.
    - 게임별 비교를 위해 게임당 리뷰 수를 어느 정도 제한한다.
    - 제한하지 않으려면 None으로 설정

- MAX_TOTAL_REVIEWS = 12
    - 전체 LLM 분석 리뷰 수의 상한
    - 예:\
    REVIEWS_PER_GAME=20이고 게임이 30개면 최대 600개가 될 수 있다.
    이때 MAX_TOTAL_REVIEWS=300이면 전체에서 300개만 뽑는다.
    - 제한하지 않으려면 None으로 설정

- SAMPLE_MODE = "balanced_by_steam_label"
    - 샘플링 방식
        - "recent"\
        최신 리뷰 우선\
        최근 패치/최근 반응을 보고 싶을 때 사용
        
        - "random"\
        무작위 샘플링\
        전체적인 평균 반응을 보고 싶을 때 사용
        
        - "balanced_by_steam_label"\
        긍정/부정 리뷰가 가능하면 섞이도록 샘플링\
        부정 원인과 긍정 요인을 함께 보고 싶을 때 추천

In [5]:
TEST_N = None
RANDOM_STATE = 42

REVIEWS_PER_GAME = 12
MAX_TOTAL_REVIEWS = None

SAMPLE_MODE = "balanced_by_steam_label"

## 1-3. LLM 호출 옵션

- BATCH_SIZE = 3
    - 한 번의 LLM 요청에 보낼 리뷰 수

    - 작게 설정하는 경우:\
    오류가 났을 때 손실이 적다.\
    하지만 요청 수가 늘어난다.

    - 크게 설정하면:\
    요청 수는 줄어든다.\
    하지만 프롬프트가 길어져 오류 가능성이 커질 수 있다.

- MAX_CONCURRENT = 1
    - 동시에 실행할 LLM 요청 개수
    - 예:
        - 1: 가장 안전하지만 느림
        - 2~3: 적당히 빠름
        - 너무 크게 잡으면 API 제한이나 오류가 날 수 있음

- MAX_RETRIES = 3
    - 요청 실패 시 재시도 횟수
    - 일시적인 API 오류나 네트워크 오류에 대비하기 위한 설정

- REQUEST_SLEEP_SEC = 1
    - chunk 단위 실행 후 잠깐 쉬는 시간
    - API 호출을 너무 빠르게 몰아서 보내지 않기 위한 안전장치

- CHUNK_SIZE = MAX_CONCURRENT * 3
    - 비동기 작업을 몇 개씩 묶어서 실행할지 정하는 값
    - 보통 MAX_CONCURRENT의 2~3배 정도로 두면 무난하다.

- MIN_REVIEW_LEN = 30
    - 너무 짧은 리뷰를 제외하기 위한 최소 글자 수
    - 예:\
    "good", "bad", "ㅋㅋㅋ" 같은 짧은 리뷰는 LLM이 과잉 해석할 수 있다.\
    그래서 일정 길이 미만은 분석 대상에서 제외한다.

MAX_REVIEW_CHARS = 1200
- 너무 긴 리뷰는 앞부분만 LLM에 보낸다.
    - 필요한 이유:
    - 긴 리뷰는 토큰 비용이 커진다.
    - 너무 긴 입력은 API 오류 가능성이 올라간다.

In [6]:
BATCH_SIZE = 3
MAX_CONCURRENT = 1
MAX_RETRIES = 3
REQUEST_SLEEP_SEC = 1

CHUNK_SIZE = MAX_CONCURRENT * 3

MIN_REVIEW_LEN = 40
MAX_REVIEW_CHARS = 1200

## 1-4. 비용 추정 옵션

In [7]:
# 아래 값은 실제 과금과 정확히 일치하지 않을 수 있다.
# 대략 어느 정도 비용이 나올지 감을 잡기 위한 참고용

INPUT_PRICE_PER_1M = 0.25       # 입력 토큰 100만 개당 예상 비용, USD 기준
OUTPUT_PRICE_PER_1M = 0.50      # 출력 토큰 100만 개당 예상 비용, USD 기준
USD_TO_KRW = 1500               # 원화 환산용 임시 환율

## 1-5. 분석 대상 필터 설정

예시:

### 예시 A. 영어 부정 리뷰만 보기

```python
ANALYSIS_FILTERS = {
    "genres": [],
    "tags": [],
    "languages": ["english"],
    "steam_labels": ["negative"],
    "review_date_from": None,
    "review_date_to": None,
    "release_periods": [],
}
```

### 예시 B. Action 장르의 출시 초기 부정 리뷰 보기

```python
ANALYSIS_FILTERS = {
    "genres": ["Action"],
    "tags": [],
    "languages": ["english"],
    "steam_labels": ["negative"],
    "release_periods": ["D0-D7", "D8-D30"],
}
```

### 예시 C. Deckbuilding 태그 게임군의 긍정/부정 비교

```python
ANALYSIS_FILTERS = {
    "genres": [],
    "tags": ["Deckbuilding"],
    "languages": ["english"],
    "steam_labels": ["positive", "negative"],
    "release_periods": [],
}
```

### 예시 D. 특정 게임 하나만 자세히 보기

```python
ANALYSIS_FILTERS = {
    "appids": [1948800],
    "game_name_contains": [],
    "genres": [],
    "tags": [],
    "languages": ["english"],
    "steam_labels": ["positive", "negative"],
}
```

기본 원칙은 다음과 같다.

- 빈 리스트(`[]`) 또는 `None`이면 해당 조건은 적용하지 않는다.
- 여러 조건을 동시에 넣으면 `AND` 조건처럼 분석 대상이 점점 좁혀진다.
- 예를 들어 `genres=["Action"]`, `languages=["english"]`, `steam_labels=["negative"]`를 함께 넣으면 Action 장르의 영어 부정 리뷰만 분석한다.

| 필터 묶음 | 설정값 | 의미 |
| --- | --- | --- |
| 게임 직접 지정 | `appids` | 특정 Steam 게임 ID만 분석한다. |
| 게임 직접 지정 | `game_name_contains` | 게임명에 특정 단어가 들어간 게임만 분석한다. |
| 게임 속성 | `genres` | Steam 공식 장르 기준으로 필터링한다. |
| 게임 속성 | `tags` | Steam 태그 기준으로 필터링한다. 메타 데이터에 포함된 태그 정보를 사용한다. |
| 게임 속성 | `release_date_from`, `release_date_to` | 게임 출시일 기준으로 필터링한다. |
| 리뷰 기준 | `languages` | 리뷰 언어 기준으로 필터링한다. |
| 리뷰 기준 | `steam_labels` | Steam 추천/비추천 라벨 기준으로 필터링한다. |
| 리뷰 기준 | `review_date_from`, `review_date_to` | 리뷰 작성일 기준으로 필터링한다. |
| 리뷰 기준 | `release_periods` | 출시일 기준 리뷰 작성 구간으로 필터링한다. |
| 리뷰 신뢰도 | `steam_purchase_only` | Steam에서 직접 구매한 유저의 리뷰만 사용할지 정한다. |
| 리뷰 신뢰도 | `exclude_received_for_free` | 무료로 받은 유저의 리뷰를 제외할지 정한다. |
| 리뷰 신뢰도 | `exclude_early_access_reviews` | 얼리액세스 기간에 작성된 리뷰를 제외할지 정한다. |

In [8]:
# ============================================================
# 본 분석 1차 대상 장르
# ============================================================
TARGET_GENRE = "Action"
TARGET_STRATA = [
    f"{TARGET_GENRE}_low",
    f"{TARGET_GENRE}_mid",
    f"{TARGET_GENRE}_high",
]

ANALYSIS_FILTERS = {
    # 게임 직접 지정
    "appids": [],
    "game_name_contains": [],

    # 표본 설계 기준 장르
    # genres보다 stratum 기준을 우선 사용
    "strata": TARGET_STRATA,

    # 게임속성
    "genres": [],
    "categories": [],
    "tags": [],
    "release_date_from": "2024-01-01",
    "release_date_to": None,

    # 리뷰 기준
    "languages": ["english"],
    "steam_labels": ["positive", "negative"],
    "review_date_from": None,
    "review_date_to": None,

    # 출시 초기 반응 중심
    "release_periods": ["D0-D7", "D8-D30"],

    # 리뷰 신뢰도
    "steam_purchase_only": True,
    "exclude_received_for_free": True,

    # 출시 직후 반응 분석이므로 얼리액세스 리뷰 제외
    "exclude_early_access_reviews": True,
}

# 2. 공통 헬퍼 함수

## 2-1. JSON 저장 보조 함수

In [9]:
def to_serializable(obj):
    """
    JSON으로 저장할 수 없는 타입을 JSON 저장 가능한 타입으로 바꿔주는 함수

    왜 필요한가?
    - pandas/numpy 타입은 사람이 보기에는 숫자나 날짜처럼 보여도, json.dump()가 바로 저장하지 못하는 경우가 많음
    - 예를 들어 np.int64, np.float64, pd.Timestamp 같은 값은 JSON 기본 타입이 아니라서 저장 중 TypeError가 날 수 있다.
    - 그래서 결과 저장 전에 모든 값을 int, float, bool, str, list, dict, None 같은 JSON 친화적인 타입으로 변환
    """
    # obj가 딕셔너리라면, 딕셔너리 안의 값들을 하나씩 다시 변환
    # 예: {"a": np.int64(1)} -> {"a": 1}
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}

    # obj가 리스트라면, 리스트 안의 원소들을 하나씩 다시 변환
    # 예: [np.int64(1), np.float64(2.5)] -> [1, 2.5]
    elif isinstance(obj, list):
        return [to_serializable(v) for v in obj]

    # obj가 튜플이라면 JSON에는 튜플 타입이 없으므로 리스트로 변환
    # 예: (1, 2) -> [1, 2]
    elif isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]

    # numpy의 정수 타입은 Python 기본 int로 변환
    # json.dump()는 np.int64를 직접 저장하지 못할 수 있다.
    elif isinstance(obj, np.integer):
        return int(obj)

    # numpy의 실수 타입은 Python 기본 float로 변환
    elif isinstance(obj, np.floating):
        # np.nan은 JSON에서 안전하게 다루기 어렵기 때문에 None으로 바꿈
        # None은 JSON 저장 시 null로 저장
        if np.isnan(obj):
            return None
        return float(obj)

    # numpy의 bool 타입은 Python 기본 bool로 변환
    # 예: np.bool_(True) -> True
    elif isinstance(obj, np.bool_):
        return bool(obj)

    # pandas Timestamp는 ISO 형식 문자열로 변환
    # 예: 2026-04-24 10:00:00 -> "2026-04-24T10:00:00"
    elif isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    # Python datetime 객체도 ISO 형식 문자열로 변환
    elif isinstance(obj, datetime):
        return obj.isoformat()

    # pandas 기준 결측치라면 None으로 변환
    # 예: NaN, NaT -> None
    elif pd.isna(obj):
        return None
    
    # 위 조건에 해당하지 않는 값은 그대로 반환
    else:
        return obj
    
# ai가 내주는 솔루션, 왜 작동 가능한지 나도 몰?루

## 2-2. 리스트 / 태그 파싱 함수

In [10]:
def parse_list_like(value):
    """
    문자열로 저장된 리스트를 Python 리스트로 바꾸는 함수

    예:
    - "['Action', 'Indie']" -> ['Action', 'Indie']
    - "Single-player, Co-op, Steam Achievements" -> ['Single-player', 'Co-op', 'Steam Achievements']
    - "Action" -> ['Action']
    - 결측/파싱 실패 -> []
    """
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "":
        return []

    # Python 리스트 문자열 처리
    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]

        if isinstance(parsed, tuple):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    # 콤마 구분 문자열 처리
    # 현재 steam_indie_games.csv의 categories는 이 형태로 저장되어 있다.
    if "," in text:
        return [x.strip() for x in text.split(",") if x.strip()]

    # 단일 문자열 처리
    return [text]


In [11]:
def parse_tag_dict(value):
    """
    tags 컬럼의 JSON 문자열을 dict로 변환한다.

    예:
    '{"RPG": 103, "Puzzle": 129}' -> {"RPG": 103, "Puzzle": 129}

    태그명만 필요한 경우도 있지만, 투표수/가중치가 필요할 수 있으므로 dict 형태도 보존한다.
    """
    if isinstance(value, dict):
        return value

    if pd.isna(value):
        return {}

    text = str(value).strip()
    if text == "":
        return {}

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            cleaned = {}
            for k, v in parsed.items():
                key = str(k).strip()
                try:
                    val = float(v)
                except Exception:
                    val = np.nan
                if key:
                    cleaned[key] = val
            return cleaned
    except Exception:
        pass

    return {}

In [12]:
def parse_tag_keys(value):
    """
    tags 컬럼의 JSON 문자열에서 태그명만 리스트로 꺼내는 함수

    예:
    - '{"RPG": 442, "Magic": 324}' -> ['RPG', 'Magic']
    - 결측/파싱 실패 -> []
    """
    if isinstance(value, dict):
        return [str(k).strip() for k in value.keys()]
    if pd.isna(value):
        return []
    try:
        parsed = json.loads(str(value))
        if isinstance(parsed, dict):
            return [str(k).strip() for k in parsed.keys()]
    except Exception:
        pass
    return []

In [13]:
def get_top_tags(value, top_n=10):
    """
    태그 투표수 기준 상위 태그만 추출한다.

    LLM 프롬프트에 태그를 너무 많이 넣으면 길어질 수 있으므로,
    프롬프트에는 상위 태그만 넣는 것이 안전하다.
    """
    tag_dict = parse_tag_dict(value)
    if not tag_dict:
        return []

    sorted_items = sorted(
        tag_dict.items(),
        key=lambda x: (-1 if pd.isna(x[1]) else -x[1], x[0])
    )
    return [k for k, _ in sorted_items[:top_n]]


In [14]:
def list_to_text(values):
    """리스트를 프롬프트와 표에서 보기 좋은 문자열로 변환"""
    if not isinstance(values, list) or len(values) == 0:
        return ""
    return ", ".join(str(x) for x in values)

In [15]:
def contains_any(values, targets):
    """
    리스트 values 안에 targets 중 하나라도 들어 있는지 확인
    대소문자 차이 때문에 놓치지 않도록 모두 소문자로 비교
    """
    if not targets:
        return True
    if not isinstance(values, list):
        return False

    value_set = {str(x).strip().lower() for x in values}
    target_set = {str(x).strip().lower() for x in targets}
    return len(value_set & target_set) > 0

## 2-3. 플레이타임 / 출시 구간 생성 함수

In [16]:
def get_playtime_stage(hours):
    """
    리뷰 작성 시점 플레이타임을 구간으로 나누는 함수

    목적:
    - 같은 부정 리뷰라도 10분 플레이 후 남긴 리뷰와 20시간 플레이 후 남긴 리뷰는 의미가 다를 수 있다.
    - 플레이타임 구간을 함께 넘기면 LLM이 리뷰 맥락을 해석하는 데 도움이 됨

    기준:
    - 30분 미만: very_early
    - 30분 이상 2시간 미만: early
    - 2시간 이상 10시간 미만: mid
    - 10시간 이상: late
    """
    if pd.isna(hours):
        return "unknown"
    if hours < 0.5:
        return "very_early"
    elif hours < 2:
        return "early"
    elif hours < 10:
        return "mid"
    else:
        return "late"

In [17]:
def get_release_period(days):
    """
    출시일 기준 리뷰 작성 시점을 구간화하는 함수

    목적:
    - 출시 직후 반응과 장기 반응을 구분하기 위함
    - 초기 반응 분석에서는 D0-D7, D8-D30 같은 구간이 중요함
    """
    if pd.isna(days):
        return "unknown"

    if days < 0:
        return "pre_release"
    elif days <= 7:
        return "D0-D7"
    elif days <= 30:
        return "D8-D30"
    elif days <= 90:
        return "D31-D90"
    elif days <= 180:
        return "D91-D180"
    else:
        return "D181+"

## 2-4. 점검 로그 / 체크포인트 / 비용 출력 함수

In [18]:
def print_filter_step(log_rows, step_name, before_count, after_count):
    """필터 적용 전후 행 수를 보기 좋게 출력하고 로그로 남기는 함수"""
    removed = before_count - after_count
    log_rows.append({
        "step": step_name,
        "before_count": before_count,
        "after_count": after_count,
        "removed_count": removed,
    })
    print(f"{step_name}: {before_count:,} -> {after_count:,} / 제거 {removed:,}")


In [19]:
def load_checkpoint(path=CHECKPOINT_PATH):
    """
    이전에 저장된 checkpoint 결과를 불러오는 함수

    목적:
    - LLM 분석 중간에 노트북이 중단되거나 API 오류가 나도,
      이미 처리한 리뷰를 다시 호출하지 않기 위함
    """
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_checkpoint(results, path=CHECKPOINT_PATH):
    """
    현재까지의 LLM 분석 결과를 checkpoint JSON으로 저장하는 함수

    목적:
    - 배치가 끝날 때마다 결과를 저장해두면,
      중간에 실행이 끊겨도 이어서 분석할 수 있다.
    """
    safe_results = to_serializable(results)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)

In [20]:
def load_existing_results():
    """
    RUN_LLM=False일 때 기존 결과를 읽기 위한 함수

    우선순위:
    1. 최종 결과 JSON
    2. checkpoint JSON
    3. 둘 다 없으면 빈 리스트
    """
    if RESULT_JSON_PATH.exists():
        with open(RESULT_JSON_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return load_checkpoint(CHECKPOINT_PATH)

In [21]:
def print_cost_report(input_tokens, output_tokens, requests, checkpoint_count, to_process_count):
    """토큰 사용량과 예상 비용을 간단히 출력하는 함수"""
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    total_cost = input_cost + output_cost

    print("=" * 60)
    print("토큰 사용량 / 예상 비용")
    print("=" * 60)
    print(f"요청 수: {requests:,}")
    print(f"checkpoint에서 불러온 리뷰 수: {checkpoint_count:,}")
    print(f"이번 실행에서 새로 처리할 리뷰 수: {to_process_count:,}")
    print(f"입력 토큰: {input_tokens:,}")
    print(f"출력 토큰: {output_tokens:,}")
    print(f"예상 비용(USD): ${total_cost:,.6f}")
    print(f"예상 비용(KRW): ₩{total_cost * USD_TO_KRW:,.0f}")
    print("=" * 60)

# 3. 데이터 로드

In [22]:
# 이 셀은 본 분석에 필요한 필수 데이터 로드 셀이다.

df_reviews_raw = pd.read_csv(INPUT_REVIEW_PATH)
df_games_raw = pd.read_csv(GAME_META_PATH)
df_hist_raw = pd.read_csv(REVIEW_HISTOGRAM_PATH)


print("리뷰 데이터:", df_reviews_raw.shape)
print("게임 메타 데이터:", df_games_raw.shape)
print("리뷰 히스토리 데이터:", df_hist_raw.shape)

리뷰 데이터: (166875, 24)
게임 메타 데이터: (9169, 20)
리뷰 히스토리 데이터: (11665, 10)


## 3-1. 확인용: 데이터 기본 구조 확인

확인용

삭제해도 작동

In [23]:
if RUN_CHECK_CELLS:
    print("리뷰 컬럼")
    display(pd.DataFrame({
        "column": df_reviews_raw.columns,
        "dtype": [df_reviews_raw[c].dtype for c in df_reviews_raw.columns],
        "missing": [df_reviews_raw[c].isna().sum() for c in df_reviews_raw.columns],
    }))

    print("게임 메타 컬럼")
    display(pd.DataFrame({
        "column": df_games_raw.columns,
        "dtype": [df_games_raw[c].dtype for c in df_games_raw.columns],
        "missing": [df_games_raw[c].isna().sum() for c in df_games_raw.columns],
    }))

    print("리뷰 히스토리 컬럼")
    display(pd.DataFrame({
        "column": df_hist_raw.columns,
        "dtype": [df_hist_raw[c].dtype for c in df_hist_raw.columns],
        "missing": [df_hist_raw[c].isna().sum() for c in df_hist_raw.columns],
    }))


리뷰 컬럼


,column,dtype,missing
0,recommendationid,int64,0
1,appid,int64,0
2,language,str,0
3,review,str,0
4,timestamp_created,int64,0
5,timestamp_updated,int64,0
6,voted_up,bool,0
7,votes_up,int64,0
8,votes_funny,int64,0
9,weighted_vote_score,float64,0


게임 메타 컬럼


,column,dtype,missing
0,appid,int64,0
1,positive,int64,0
2,negative,int64,0
3,price,float64,0
4,genres,str,0
5,total_reviews,int64,0
6,name,str,0
7,developers,str,0
8,release_date,str,344
9,short_description,str,107


리뷰 히스토리 컬럼


,column,dtype,missing
0,appid,int64,0
1,name,str,0
2,release_date,str,495
3,hist_start_date,str,0
4,hist_end_date,str,0
5,date,str,0
6,recommendations_up,int64,0
7,recommendations_down,int64,0
8,data_type,str,0
9,recommendations_total,int64,0


# 4. 데이터 전처리

## 4-1. 리뷰 데이터 정리

In [24]:
df_reviews = df_reviews_raw.copy()

# ============================================================
# 1. 키 컬럼 정리
# ============================================================
# recommendationid는 리뷰 고유 ID다.
# 숫자로 보여도 JSON 저장/LLM 결과 매칭에서는 문자열이 더 안전하다.
df_reviews["recommendationid"] = df_reviews["recommendationid"].astype(str)

# appid는 게임 메타/히스토그램과 조인할 때 사용하는 키다.
df_reviews["appid"] = pd.to_numeric(df_reviews["appid"], errors="coerce").astype("Int64")

# ============================================================
# 2. 날짜 컬럼 정리
# ============================================================
# created_date가 이미 있으면 우선 사용하고,
# 없거나 변환 실패한 경우 timestamp_created를 Unix timestamp로 변환한다.
if "created_date" in df_reviews.columns:
    df_reviews["review_datetime"] = pd.to_datetime(df_reviews["created_date"], errors="coerce")
else:
    df_reviews["review_datetime"] = pd.NaT

if "timestamp_created" in df_reviews.columns:
    fallback_created = pd.to_datetime(df_reviews["timestamp_created"], unit="s", errors="coerce")
    df_reviews["review_datetime"] = df_reviews["review_datetime"].fillna(fallback_created)

# 수정일도 같은 방식으로 정리한다.
if "updated_date" in df_reviews.columns:
    df_reviews["review_updated_datetime"] = pd.to_datetime(df_reviews["updated_date"], errors="coerce")
else:
    df_reviews["review_updated_datetime"] = pd.NaT

if "timestamp_updated" in df_reviews.columns:
    fallback_updated = pd.to_datetime(df_reviews["timestamp_updated"], unit="s", errors="coerce")
    df_reviews["review_updated_datetime"] = df_reviews["review_updated_datetime"].fillna(fallback_updated)

# ============================================================
# 3. Steam 추천/비추천 라벨 생성
# ============================================================
# voted_up=True  -> positive
# voted_up=False -> negative
# 이 라벨은 Steam 원본 라벨이고, LLM이 판단하는 감정과는 별개다.
df_reviews["steam_label_text"] = np.where(
    df_reviews["voted_up"].astype(bool),
    "positive",
    "negative"
)

# ============================================================
# 4. 리뷰 본문 정리
# ============================================================
# LLM에 입력할 텍스트는 반드시 문자열이어야 한다.
df_reviews["review"] = df_reviews["review"].fillna("").astype(str)

# 리뷰 길이는 너무 짧은 리뷰를 제외할 때 사용한다.
df_reviews["review_len"] = df_reviews["review"].str.len()

# 너무 긴 리뷰는 LLM 비용과 오류 방지를 위해 앞부분만 사용한다.
df_reviews["review_text_for_llm"] = df_reviews["review"].str.slice(0, MAX_REVIEW_CHARS)

# ============================================================
# 5. 플레이타임 컬럼 정리
# ============================================================
# 이번 데이터에는 playtime_at_review_hours가 있다.
# 과거 코드의 author_playtime_at_review는 현재 파일에 없으므로 여기서 호환 컬럼을 만들어준다.
if "playtime_at_review_hours" in df_reviews.columns:
    df_reviews["playtime_at_review_hours"] = pd.to_numeric(
        df_reviews["playtime_at_review_hours"], errors="coerce"
    )
elif "author_playtime_at_review" in df_reviews.columns:
    # 과거 Steam API 원본 구조에서는 분 단위일 가능성이 높다.
    df_reviews["playtime_at_review_hours"] = pd.to_numeric(
        df_reviews["author_playtime_at_review"], errors="coerce"
    ) / 60
else:
    df_reviews["playtime_at_review_hours"] = np.nan

if "playtime_forever_hours" in df_reviews.columns:
    df_reviews["playtime_forever_hours"] = pd.to_numeric(
        df_reviews["playtime_forever_hours"], errors="coerce"
    )
else:
    df_reviews["playtime_forever_hours"] = np.nan

# 플레이타임을 구간으로 변환한다.
df_reviews["playtime_stage"] = df_reviews["playtime_at_review_hours"].apply(get_playtime_stage)

print("리뷰 데이터 정리 완료:", df_reviews.shape)

리뷰 데이터 정리 완료: (166875, 30)


### 4-1-1. 확인용: 리뷰 전처리 결과 점검
확인용이다. 
삭제해도 작동한다.

In [25]:
if RUN_CHECK_CELLS:
    print("리뷰 언어 분포")
    display(df_reviews["language"].value_counts().head(15))

    print("Steam 라벨 분포")
    display(df_reviews["steam_label_text"].value_counts())

    print("플레이타임 구간 분포")
    display(df_reviews["playtime_stage"].value_counts(dropna=False))

    display(
        df_reviews[
            [
                "recommendationid", "appid", "language", "steam_label_text",
                "review_datetime", "playtime_at_review_hours", "playtime_stage",
                "review_len", "review_text_for_llm"
            ]
        ].head()
    )

리뷰 언어 분포


language
english      76272
schinese     32538
russian      14165
spanish       8033
german        7000
french        5600
koreana       5144
brazilian     3877
japanese      3058
tchinese      2557
polish        2016
czech         1048
latam         1008
turkish        918
italian        879
Name: count, dtype: int64

Steam 라벨 분포


steam_label_text
positive    151042
negative     15833
Name: count, dtype: int64

플레이타임 구간 분포


playtime_stage
late          82598
mid           66129
early         13710
very_early     4438
Name: count, dtype: int64

,recommendationid,appid,language,steam_label_text,review_datetime,playtime_at_review_hours,playtime_stage,review_len,review_text_for_llm
0,18698790,324470,french,positive,2015-10-26 18:10:33,1.250000,early,24,good game for this price
1,18699465,324470,english,positive,2015-10-26 18:53:36,0.216667,very_early,119,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ..."
2,18699648,324470,english,positive,2015-10-26 19:03:37,12.666667,late,387,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re..."
3,18700348,324470,english,positive,2015-10-26 19:52:39,0.916667,early,1100,Ever played Bhop? Surf? If so this games mechanics will feel Instantly similar too you. This game gives you such a r...
4,18701774,324470,english,positive,2015-10-26 21:32:24,6.416667,mid,8,It's Lit


## 4-2. 게임 메타 데이터 정리

In [26]:
df_games = df_games_raw.copy()

# ============================================================
# 1. 키/이름 컬럼 정리
# ============================================================
df_games["appid"] = pd.to_numeric(df_games["appid"], errors="coerce").astype("Int64")

if "name" not in df_games.columns:
    df_games["name"] = df_games["appid"].astype(str)

df_games["game_name"] = df_games["name"].fillna(df_games["appid"].astype(str)).astype(str)

# ============================================================
# 2. 장르/카테고리/태그 정리
# ============================================================
# genres, categories는 콤마 구분 문자열이므로 리스트로 바꾼다.
df_games["genres_list"] = df_games["genres"].apply(parse_list_like) if "genres" in df_games.columns else [[] for _ in range(len(df_games))]
df_games["categories_list"] = df_games["categories"].apply(parse_list_like) if "categories" in df_games.columns else [[] for _ in range(len(df_games))]

# tags는 JSON 문자열 형태다.
# 전체 태그 리스트와 상위 태그 리스트를 둘 다 만든다.
if "tags" in df_games.columns:
    df_games["steam_tags_dict"] = df_games["tags"].apply(parse_tag_dict)
    df_games["steam_tags_list"] = df_games["tags"].apply(parse_tag_keys)
    df_games["top_steam_tags_list"] = df_games["tags"].apply(lambda x: get_top_tags(x, top_n=10))
else:
    df_games["steam_tags_dict"] = [{} for _ in range(len(df_games))]
    df_games["steam_tags_list"] = [[] for _ in range(len(df_games))]
    df_games["top_steam_tags_list"] = [[] for _ in range(len(df_games))]

# 보기 좋은 문자열 컬럼
df_games["genres_text"] = df_games["genres_list"].apply(list_to_text)
df_games["categories_text"] = df_games["categories_list"].apply(list_to_text)
df_games["steam_tags_text"] = df_games["steam_tags_list"].apply(list_to_text)
df_games["top_steam_tags_text"] = df_games["top_steam_tags_list"].apply(list_to_text)

# ============================================================
# 3. 출시일/숫자형 컬럼 정리
# ============================================================
df_games["release_date"] = pd.to_datetime(df_games["release_date"], errors="coerce") if "release_date" in df_games.columns else pd.NaT

numeric_cols = [
    "price", "positive", "negative", "total_reviews", "review_score",
    "recommendations_total", "achievements_total", "owners_lower", "owners_higher"
]

for col in numeric_cols:
    if col in df_games.columns:
        df_games[col] = pd.to_numeric(df_games[col], errors="coerce")

# 가격대는 보고서/필터링에서 참고할 수 있도록 구간화한다.
# 현재 CSV의 price 컬럼은 이미 14.99, 24.99처럼 USD 단위로 저장되어 있으므로 /100을 하지 않는다.
if "price" in df_games.columns:
    df_games["price_usd"] = pd.to_numeric(df_games["price"], errors="coerce")

    df_games["price_group"] = pd.cut(
        df_games["price_usd"].fillna(-1),
        bins=[-2, -0.1, 0, 5, 10, 20, 40, np.inf],
        labels=["unknown", "free", "0-5", "5-10", "10-20", "20-40", "40+"]
    ).astype(str)
else:
    df_games["price_usd"] = np.nan
    df_games["price_group"] = "unknown"

# 누적 긍정률 계산
if {"positive", "negative"}.issubset(df_games.columns):
    total = df_games["positive"] + df_games["negative"]
    df_games["meta_positive_ratio"] = np.where(total > 0, df_games["positive"] / total, np.nan)
else:
    df_games["meta_positive_ratio"] = np.nan

# ============================================================
# 4. 조인에 필요한 컬럼만 추림
# ============================================================
game_cols = [
    "appid", "game_name", "developers", "publishers", "short_description",
    "genres_list", "genres_text", "categories_list", "categories_text",
    "steam_tags_list", "steam_tags_text", "top_steam_tags_list", "top_steam_tags_text",
    "release_date", "price", "price_group", "positive", "negative", "total_reviews",
    "meta_positive_ratio", "review_score", "review_score_desc",
    "recommendations_total", "achievements_total", "owners_lower", "owners_higher",
    "windows", "mac", "linux"
]

game_cols = [c for c in game_cols if c in df_games.columns]
df_games_meta = df_games[game_cols].drop_duplicates("appid")

print("게임 메타 정리 완료:", df_games_meta.shape)

게임 메타 정리 완료: (9169, 27)


### 4-1-2. 확인용: 게임 메타 전처리 결과 점검
확인용이다. 
삭제해도 작동한다.

In [27]:
if RUN_CHECK_CELLS:
    print("장르 분포 상위 20개")
    genre_counts = (
        df_games_meta[["appid", "genres_list"]]
        .explode("genres_list")
        .dropna(subset=["genres_list"])
        ["genres_list"]
        .value_counts()
        .head(20)
    )
    display(genre_counts)

    print("태그 분포 상위 30개")
    tag_counts = (
        df_games_meta[["appid", "steam_tags_list"]]
        .explode("steam_tags_list")
        .dropna(subset=["steam_tags_list"])
        ["steam_tags_list"]
        .value_counts()
        .head(30)
    )
    display(tag_counts)

    display(
        df_games_meta[
            [
                "appid", "game_name", "release_date", "genres_text",
                "categories_text", "top_steam_tags_text", "price", "total_reviews"
            ]
        ].head()
    )

장르 분포 상위 20개


genres_list
Indie         9164
Adventure     4732
Casual        4051
Action        3995
Simulation    2429
RPG           2155
Strategy      1980
Sports         329
Racing         284
Name: count, dtype: int64

태그 분포 상위 30개


steam_tags_list
Singleplayer            7077
Indie                   5501
Adventure               4414
Casual                  3996
Action                  3848
2D                      3837
3D                      2981
Atmospheric             2688
Exploration             2594
Simulation              2347
Story Rich              2282
Cute                    2228
Colorful                2219
Pixel Graphics          2197
Puzzle                  2160
First-Person            2121
RPG                     2109
Strategy                1921
Horror                  1786
Action-Adventure        1712
Fantasy                 1662
Relaxing                1547
Funny                   1478
Controller              1421
Psychological Horror    1414
Stylized                1388
Arcade                  1306
Combat                  1256
Retro                   1241
Dark                    1211
Name: count, dtype: int64

,appid,game_name,release_date,genres_text,categories_text,top_steam_tags_text,price,total_reviews
0,226620,Desktop Dungeons,2023-04-18,"Adventure, Casual, Indie, RPG, Strategy","Single-player, Steam Achievements, Steam Trading Cards, Family Sharing","Rogue-like, Turn-Based, Puzzle, RPG, Singleplayer, Fantasy, Resource Management, Difficult, Casual, Great Soundtrack",14.99,2276
1,230210,ASYLUM,2025-03-13,"Adventure, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Family Sharing","Adventure, Indie, Horror, Point & Click, Psychological Horror, Atmospheric, Gore, First-Person, Mystery, Story Rich",24.99,348
2,251570,7 Days to Die,2024-07-25,"Action, Adventure, Indie, RPG, Simulation, Strategy","Single-player, Multi-player, PvP, Online PvP, LAN PvP, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, S...","Survival, Zombies, Multiplayer, Open World, Open World Survival Craft, Base-Building, Post-apocalyptic, Voxel, Onlin...",44.99,370046
3,252190,Defender's Quest 2: Mists of Ruin,2025-01-30,"Indie, RPG, Strategy","Single-player, Steam Achievements, Full controller support, Stats, Family Sharing","Strategy, RPG, Tower Defense, Tactical RPG, 2D, Female Protagonist, Real Time Tactics, Indie, Strategy RPG, Tactical",19.99,255
4,269770,Secrets of Grindea,2024-02-29,"Action, Adventure, Indie, RPG","Single-player, Multi-player, Co-op, Online Co-op, Steam Achievements, Full controller support, Steam Cloud, Steam Le...","Adventure, Singleplayer, Multiplayer, Indie, Action RPG, JRPG, Co-op Campaign, Character Customization, RPG, Action-...",14.99,8270


## 4-3. 리뷰 히스토그램 데이터 전처리

In [28]:
df_hist = df_hist_raw.copy()

# ============================================================
# 1. 키/날짜 컬럼 정리
# ============================================================
df_hist["appid"] = pd.to_numeric(df_hist["appid"], errors="coerce").astype("Int64")

for col in ["release_date", "hist_start_date", "hist_end_date", "date"]:
    if col in df_hist.columns:
        df_hist[col] = pd.to_datetime(df_hist[col], errors="coerce")

# ============================================================
# 2. 숫자형 컬럼 정리
# ============================================================
for col in ["recommendations_up", "recommendations_down", "recommendations_total"]:
    if col in df_hist.columns:
        df_hist[col] = pd.to_numeric(df_hist[col], errors="coerce").fillna(0)

# 수정: 업로드된 histogram 파일에 recommendations_total이 없으면 up + down으로 생성한다.
if "recommendations_total" not in df_hist.columns:
    df_hist["recommendations_total"] = df_hist["recommendations_up"] + df_hist["recommendations_down"]

# ============================================================
# 3. 히스토그램 날짜의 출시일 기준 구간 생성
# ============================================================
# 리뷰 본문 데이터의 review_datetime과는 별개로,
# 날짜별 리뷰 증가량도 출시일 기준 D0-D7, D8-D30 등으로 나눌 수 있다.
df_hist["hist_days_from_release"] = (
    df_hist["date"].dt.normalize() - df_hist["release_date"].dt.normalize()
).dt.days

df_hist["hist_release_period"] = df_hist["hist_days_from_release"].apply(get_release_period)

# ============================================================
# 4. 게임 단위 히스토그램 요약 생성
# ============================================================
df_hist_game_summary = (
    df_hist
    .groupby("appid", as_index=False)
    .agg(
        hist_game_name=("name", "first"),
        stratum=("stratum", "first"),
        hist_first_date=("date", "min"),
        hist_last_date=("date", "max"),
        hist_positive_reviews=("recommendations_up", "sum"),
        hist_negative_reviews=("recommendations_down", "sum"),
        hist_total_reviews=("recommendations_total", "sum"),
        hist_data_type_count=("data_type", "nunique"),
    )
)

df_hist_game_summary["hist_positive_ratio"] = np.where(
    df_hist_game_summary["hist_total_reviews"] > 0,
    df_hist_game_summary["hist_positive_reviews"] / df_hist_game_summary["hist_total_reviews"],
    np.nan
)

# ============================================================
# 5. 게임 x 출시구간 요약 생성
# ============================================================
# 이 표는 나중에 D0-D7, D8-D30 리뷰 증가량을 보고 싶을 때 사용한다.
df_hist_period_summary = (
    df_hist
    .groupby(["appid", "hist_release_period"], as_index=False)
    .agg(
        period_positive_reviews=("recommendations_up", "sum"),
        period_negative_reviews=("recommendations_down", "sum"),
        period_total_reviews=("recommendations_total", "sum"),
    )
)

df_hist_period_summary["period_positive_ratio"] = np.where(
    df_hist_period_summary["period_total_reviews"] > 0,
    df_hist_period_summary["period_positive_reviews"] / df_hist_period_summary["period_total_reviews"],
    np.nan
)

print("히스토그램 정리 완료:", df_hist.shape)
print("게임 단위 히스토그램 요약:", df_hist_game_summary.shape)
print("게임 x 출시구간 히스토그램 요약:", df_hist_period_summary.shape)

KeyError: "Label(s) ['stratum'] do not exist"

### 확인용: 히스토그램 전처리 결과 점검
확인용이다. 
삭제해도 작동한다.

In [ ]:
if RUN_CHECK_CELLS:
    print("히스토그램 data_type 분포")
    display(df_hist["data_type"].value_counts(dropna=False))

    print("히스토그램 stratum 분포 상위 20개")
    display(df_hist["stratum"].value_counts().head(20))

    print("히스토그램 출시 구간 분포")
    display(df_hist["hist_release_period"].value_counts(dropna=False))

    display(df_hist_game_summary.head())
    display(df_hist_period_summary.head())

히스토그램 data_type 분포


data_type
recent     5947
rollups    5668
Name: count, dtype: int64

히스토그램 stratum 분포 상위 20개


stratum
Action_low         859
Action_mid         825
Strategy_mid       781
Strategy_high      718
RPG_mid            716
Sports_high        569
Strategy_low       569
Simulation_mid     565
Casual_low         543
Action_high        482
RPG_high           451
Simulation_low     425
RPG_low            423
Casual_mid         419
Simulation_high    411
Adventure_mid      402
Racing_high        391
Casual_high        369
Racing_mid         358
Adventure_high     330
Name: count, dtype: int64

히스토그램 출시 구간 분포


hist_release_period
D181+          8416
pre_release     894
D91-D180        860
D31-D90         575
unknown         374
D8-D30          303
D0-D7           193
Name: count, dtype: int64

,appid,hist_game_name,stratum,hist_first_date,hist_last_date,hist_positive_reviews,hist_negative_reviews,hist_total_reviews,hist_data_type_count,hist_positive_ratio
0,324470,SinaRun,Racing_mid,2015-10-01,2026-04-26,140,40,180,2,0.777778
1,571740,Golf It!,Sports_high,2017-02-01,2026-04-30,23320,2564,25884,2,0.900943
2,588440,False Front,Sports_high,2019-05-01,2026-04-14,68,24,92,2,0.739130
3,593150,Ooblets,RPG_high,2023-10-01,2026-04-23,1286,128,1414,2,0.909477
4,638990,"UNDYING - ""KINGDOM""",Adventure_high,2021-10-01,2026-04-28,1335,403,1738,2,0.768124


,appid,hist_release_period,period_positive_reviews,period_negative_reviews,period_total_reviews,period_positive_ratio
0,324470,D31-D90,3,0,3,1.000000
1,324470,D91-D180,2,3,5,0.400000
2,324470,pre_release,135,37,172,0.784884
3,571740,D181+,3601,444,4045,0.890235
4,571740,D31-D90,2201,55,2256,0.975621


## 4-4. 리뷰 + 게임 메타 + 태그 결합

In [ ]:
# ============================================================
# 1. 리뷰 + 게임 메타 결합
# ============================================================
df_base = df_reviews.merge(
    df_games_meta,
    on="appid",
    how="left"
)

# ============================================================
# 2. 리뷰 + 히스토그램 게임 요약 결합
# ============================================================
df_base = df_base.merge(
    df_hist_game_summary,
    on="appid",
    how="left"
)

# ============================================================
# 3. 결합 후 결측 보정
# ============================================================
# 게임명이 없으면 appid 문자열로 대체한다.
df_base["game_name"] = df_base["game_name"].fillna(df_base["appid"].astype(str))

# 리스트 컬럼은 결측이 생기면 이후 apply에서 오류가 날 수 있으므로 빈 리스트로 보정한다.
for col in ["genres_list", "categories_list", "steam_tags_list", "top_steam_tags_list"]:
    if col in df_base.columns:
        df_base[col] = df_base[col].apply(lambda x: x if isinstance(x, list) else [])

# 문자열 컬럼도 다시 정리한다.
df_base["genres_text"] = df_base["genres_list"].apply(list_to_text)
df_base["categories_text"] = df_base["categories_list"].apply(list_to_text)
df_base["steam_tags_text"] = df_base["steam_tags_list"].apply(list_to_text)
df_base["top_steam_tags_text"] = df_base["top_steam_tags_list"].apply(list_to_text)

# ============================================================
# 4. 출시일 기준 리뷰 작성 경과일 생성
# ============================================================
# 출시일보다 리뷰 작성일이 빠르면 pre_release로 분류된다.
# 얼리액세스 시절 리뷰가 여기에 포함될 수 있다.
df_base["days_from_release"] = (
    df_base["review_datetime"].dt.normalize() -
    df_base["release_date"].dt.normalize()
).dt.days

df_base["release_period"] = df_base["days_from_release"].apply(get_release_period)

print("결합 데이터:", df_base.shape)
print("결합 데이터 게임 수:", df_base["appid"].nunique())

결합 데이터: (166385, 67)
결합 데이터 게임 수: 196


## 4-5. 확인용: 결합 결과 점검

확인용, 삭제해도 작동

In [ ]:
if RUN_CHECK_CELLS:
    print("게임명 결측 수:", df_base["game_name"].isna().sum())
    print("출시일 결측 수:", df_base["release_date"].isna().sum())
    print("히스토그램 stratum 결측 수:", df_base["stratum"].isna().sum())

    display(
        df_base[
            [
                "recommendationid", "appid", "game_name", "language",
                "steam_label_text", "review_datetime", "release_date",
                "days_from_release", "release_period", "stratum",
                "genres_text", "top_steam_tags_text"
            ]
        ].head()
    )

    print("리뷰 언어 분포")
    display(df_base["language"].value_counts().head(15))

    print("출시 구간 분포")

게임명 결측 수: 0
출시일 결측 수: 4507
히스토그램 stratum 결측 수: 0


,recommendationid,appid,game_name,language,steam_label_text,review_datetime,release_date,days_from_release,release_period,stratum,genres_text,top_steam_tags_text
0,18698790,324470,SinaRun,french,positive,2015-10-26 18:10:33,2025-11-03,-3661.0,pre_release,Racing_mid,"Indie, Racing","Indie, Racing, Early Access, First-Person, Parkour"
1,18699465,324470,SinaRun,english,positive,2015-10-26 18:53:36,2025-11-03,-3661.0,pre_release,Racing_mid,"Indie, Racing","Indie, Racing, Early Access, First-Person, Parkour"
2,18699648,324470,SinaRun,english,positive,2015-10-26 19:03:37,2025-11-03,-3661.0,pre_release,Racing_mid,"Indie, Racing","Indie, Racing, Early Access, First-Person, Parkour"
3,18700348,324470,SinaRun,english,positive,2015-10-26 19:52:39,2025-11-03,-3661.0,pre_release,Racing_mid,"Indie, Racing","Indie, Racing, Early Access, First-Person, Parkour"
4,18701774,324470,SinaRun,english,positive,2015-10-26 21:32:24,2025-11-03,-3661.0,pre_release,Racing_mid,"Indie, Racing","Indie, Racing, Early Access, First-Person, Parkour"


리뷰 언어 분포


language
english      72268
schinese     38644
russian      15898
spanish       9269
german        5762
brazilian     4167
french        4056
japanese      2756
tchinese      2643
koreana       2548
latam         1654
turkish       1604
polish        1217
italian        591
czech          499
Name: count, dtype: int64

출시 구간 분포


# 5. 분석 조건 필터링

In [ ]:
def apply_analysis_filters(df, filters):
    """
    앞쪽 ANALYSIS_FILTERS에 입력한 조건을 실제 데이터에 적용하는 함수.

    이 함수에서 처리하는 조건:
    - 최소 리뷰 길이
    - appid 직접 지정
    - 게임명 포함 문자열
    - 장르
    - 카테고리
    - Steam 태그
    - stratum
    - 출시일
    - 리뷰 작성일
    - 리뷰 언어
    - Steam 긍정/부정 라벨
    - 출시일 기준 리뷰 구간
    - Steam 구매 여부
    - 무료 수령 제외 여부
    - 얼리액세스 리뷰 포함 여부
    - 플레이타임 범위
    """
    filtered = df.copy()
    log_rows = []

    # ============================================================
    # 0. 최소 리뷰 길이 필터
    # ============================================================
    before = len(filtered)
    filtered = filtered[filtered["review_len"] >= MIN_REVIEW_LEN]
    print_filter_step(log_rows, "최소 리뷰 길이 필터", before, len(filtered))

    # ============================================================
    # 1. 게임 직접 지정 필터
    # ============================================================
    if filters.get("appids"):
        before = len(filtered)
        appids = [int(x) for x in filters["appids"]]
        filtered = filtered[filtered["appid"].isin(appids)]
        print_filter_step(log_rows, "appid 필터", before, len(filtered))

    if filters.get("game_name_contains"):
        before = len(filtered)
        keywords = [str(x).lower() for x in filters["game_name_contains"]]
        mask = filtered["game_name"].fillna("").str.lower().apply(
            lambda x: any(keyword in x for keyword in keywords)
        )
        filtered = filtered[mask]
        print_filter_step(log_rows, "게임명 포함 필터", before, len(filtered))

    # ============================================================
    # 2. 게임 속성 필터
    # ============================================================
    if filters.get("genres"):
        before = len(filtered)
        filtered = filtered[
            filtered["genres_list"].apply(lambda values: contains_any(values, filters["genres"]))
        ]
        print_filter_step(log_rows, "장르 필터", before, len(filtered))

    if filters.get("categories"):
        before = len(filtered)
        filtered = filtered[
            filtered["categories_list"].apply(lambda values: contains_any(values, filters["categories"]))
        ]
        print_filter_step(log_rows, "카테고리 필터", before, len(filtered))

    if filters.get("tags"):
        before = len(filtered)
        filtered = filtered[
            filtered["steam_tags_list"].apply(lambda values: contains_any(values, filters["tags"]))
        ]
        print_filter_step(log_rows, "태그 필터", before, len(filtered))

    if filters.get("strata"):
        before = len(filtered)
        strata = [str(x).lower() for x in filters["strata"]]
        filtered = filtered[filtered["stratum"].fillna("").str.lower().isin(strata)]
        print_filter_step(log_rows, "stratum 필터", before, len(filtered))

    # ============================================================
    # 3. 날짜 필터
    # ============================================================
    if filters.get("release_date_from"):
        before = len(filtered)
        date_from = pd.to_datetime(filters["release_date_from"])
        filtered = filtered[filtered["release_date"] >= date_from]
        print_filter_step(log_rows, "출시일 시작 필터", before, len(filtered))

    if filters.get("release_date_to"):
        before = len(filtered)
        date_to = pd.to_datetime(filters["release_date_to"])
        filtered = filtered[filtered["release_date"] <= date_to]
        print_filter_step(log_rows, "출시일 종료 필터", before, len(filtered))

    if filters.get("review_date_from"):
        before = len(filtered)
        date_from = pd.to_datetime(filters["review_date_from"])
        filtered = filtered[filtered["review_datetime"] >= date_from]
        print_filter_step(log_rows, "리뷰 작성일 시작 필터", before, len(filtered))

    if filters.get("review_date_to"):
        before = len(filtered)
        date_to = pd.to_datetime(filters["review_date_to"])
        filtered = filtered[filtered["review_datetime"] <= date_to]
        print_filter_step(log_rows, "리뷰 작성일 종료 필터", before, len(filtered))

    if filters.get("release_periods"):
        before = len(filtered)
        periods = [str(x) for x in filters["release_periods"]]
        filtered = filtered[filtered["release_period"].isin(periods)]
        print_filter_step(log_rows, "출시 기준 리뷰 구간 필터", before, len(filtered))

    # ============================================================
    # 4. 리뷰 속성 필터
    # ============================================================
    if filters.get("languages"):
        before = len(filtered)
        languages = [str(x).lower() for x in filters["languages"]]
        filtered = filtered[filtered["language"].fillna("").str.lower().isin(languages)]
        print_filter_step(log_rows, "리뷰 언어 필터", before, len(filtered))

    if filters.get("steam_labels"):
        before = len(filtered)
        labels = [str(x).lower() for x in filters["steam_labels"]]
        filtered = filtered[filtered["steam_label_text"].str.lower().isin(labels)]
        print_filter_step(log_rows, "Steam 긍정/부정 라벨 필터", before, len(filtered))

    if filters.get("steam_purchase_only") is True and "steam_purchase" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["steam_purchase"] == True]
        print_filter_step(log_rows, "Steam 구매 리뷰 필터", before, len(filtered))

    if filters.get("exclude_received_for_free") is True and "received_for_free" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["received_for_free"] == False]
        print_filter_step(log_rows, "무료 수령 리뷰 제외 필터", before, len(filtered))
    
    if filters.get("exclude_early_access_reviews") is True and "written_during_early_access" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["written_during_early_access"] == False]
        print_filter_step(log_rows, "얼리액세스 리뷰 제외 필터", before, len(filtered))

    if filters.get("min_playtime_hours") is not None:
        before = len(filtered)
        min_hours = float(filters["min_playtime_hours"])
        filtered = filtered[filtered["playtime_at_review_hours"] >= min_hours]
        print_filter_step(log_rows, "최소 플레이타임 필터", before, len(filtered))

    if filters.get("max_playtime_hours") is not None:
        before = len(filtered)
        max_hours = float(filters["max_playtime_hours"])
        filtered = filtered[filtered["playtime_at_review_hours"] <= max_hours]
        print_filter_step(log_rows, "최대 플레이타임 필터", before, len(filtered))

    filter_log = pd.DataFrame(log_rows)
    return filtered.reset_index(drop=True), filter_log

In [ ]:
# ============================================================
# 분석 조건 적용
# ============================================================

df_filtered, filter_log = apply_analysis_filters(df_base, ANALYSIS_FILTERS)

print("최종 필터링 결과:", df_filtered.shape)

if RUN_CHECK_CELLS:
    display(filter_log)

최소 리뷰 길이 필터: 166,385 -> 90,133 / 제거 76,252
stratum 필터: 90,133 -> 11,031 / 제거 79,102
출시일 시작 필터: 11,031 -> 7,545 / 제거 3,486
출시 기준 리뷰 구간 필터: 7,545 -> 2,105 / 제거 5,440
리뷰 언어 필터: 2,105 -> 1,794 / 제거 311
Steam 긍정/부정 라벨 필터: 1,794 -> 1,794 / 제거 0
Steam 구매 리뷰 필터: 1,794 -> 1,794 / 제거 0
무료 수령 리뷰 제외 필터: 1,794 -> 1,791 / 제거 3
얼리액세스 리뷰 제외 필터: 1,791 -> 1,791 / 제거 0
최종 필터링 결과: (1791, 67)


,step,before_count,after_count,removed_count
0,최소 리뷰 길이 필터,166385,90133,76252
1,stratum 필터,90133,11031,79102
2,출시일 시작 필터,11031,7545,3486
3,출시 기준 리뷰 구간 필터,7545,2105,5440
4,리뷰 언어 필터,2105,1794,311
5,Steam 긍정/부정 라벨 필터,1794,1794,0
6,Steam 구매 리뷰 필터,1794,1794,0
7,무료 수령 리뷰 제외 필터,1794,1791,3
8,얼리액세스 리뷰 제외 필터,1791,1791,0


## 5-1. 확인용: 필터링 결과 분포
확인용, 삭제해도 작동

In [ ]:
if RUN_CHECK_CELLS:
    print("필터링 후 게임 수:", df_filtered["appid"].nunique())
    print("필터링 후 리뷰 수:", len(df_filtered))

    display(
        df_filtered.groupby(["appid", "game_name"])
        .agg(
            review_count=("recommendationid", "count"),
            positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
            negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
            first_review_date=("review_datetime", "min"),
            last_review_date=("review_datetime", "max"),
            release_date=("release_date", "first"),
            genres=("genres_text", "first"),
            top_tags=("top_steam_tags_text", "first"),
            stratum=("stratum", "first"),
        )
        .reset_index()
        .sort_values("review_count", ascending=False)
        .head(30)
    )

    print("필터링 후 release_period 분포")
    display(df_filtered["release_period"].value_counts(dropna=False))

필터링 후 게임 수: 17
필터링 후 리뷰 수: 1791


,appid,game_name,review_count,positive_count,negative_count,first_review_date,last_review_date,release_date,genres,top_tags,stratum
1,1836730,Echo Point Nova,1242,1222,20,2024-09-24 15:00:06,2024-10-24 23:38:46,2024-09-24,"Action, Adventure, Indie","FPS, Shooter, Online Co-Op, Exploration, Action, Arena Shooter, First-Person, 3D, Stylized, Adventure",Action_high
3,2193490,First Cut: Samurai Duel,295,259,36,2024-01-17 18:49:18,2024-02-16 10:52:31,2024-01-17,"Action, Indie","Side Scroller, Hack and Slash, Martial Arts, 2D Fighter, PvP, PvE, Arcade, Swordplay, Character Customization, 2D",Action_high
5,2538000,Dojo Masters,59,56,3,2024-07-17 18:35:01,2024-08-14 11:13:02,2024-07-17,"Action, Indie","Action, 2D Fighter, Arcade, 2D, Spectacle fighter, Pixel Graphics, Martial Arts, Retro, Character Customization, Combat",Action_mid
2,2008980,XENOTILT: HOSTILE PINBALL ACTION,51,49,2,2024-11-12 22:40:38,2024-12-12 05:17:53,2024-11-12,"Action, Indie","Bullet Hell, Shoot 'Em Up, Difficult, Hack and Slash, Physics, Unforgiving, Arcade, 2D, Top-Down, Cyberpunk",Action_high
0,1443620,Pirates VR: Jolly Roger,29,23,6,2025-01-14 17:07:11,2025-02-09 19:23:08,2025-01-14,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Action_mid
14,3191540,Smack Monkey,24,24,0,2025-03-07 22:21:26,2025-04-06 14:22:30,2025-03-07,"Action, Adventure, Indie","Precision Platformer, Parkour, Pixel Graphics, Funny, Nature, Adventure, Cartoony, Platformer, Difficult, Puzzle-Pla...",Action_low
11,2840210,Bureau of Contacts,20,16,4,2025-04-13 07:33:43,2025-05-02 09:52:25,2025-04-12,"Action, Adventure, Indie","Horror, Online Co-Op, Multiplayer, Co-op, Action, Adventure, Survival Horror, PvE, Exploration, Supernatural",Action_high
13,3110260,Detective Penguin,20,19,1,2025-04-16 01:21:19,2025-05-04 03:15:36,2025-04-15,"Action, Adventure, Casual, Indie","Casual, Detective, Funny, Investigation, Cute, Cartoony, Conversation, Adventure, Interactive Fiction, Dark Humor",Action_low
7,2738110,Vengeance Hunters,17,15,2,2024-10-28 14:08:44,2024-11-12 07:07:28,2024-10-27,"Action, Indie","Action, Beat 'em up, Pixel Graphics, Post-apocalyptic, Retro, Combat, Indie, Local Co-Op, Multiplayer, Arcade",Action_mid
6,2599630,Zorro and Zedd,12,9,3,2025-04-11 23:16:35,2025-04-28 21:17:34,2025-04-11,"Action, Adventure, Indie","Adventure, 3D Platformer, Dragons, Dark Fantasy, Hack and Slash, Parkour, Platformer, Action-Adventure, 3D, Third Pe...",Action_low


필터링 후 release_period 분포


release_period
D0-D7     1061
D8-D30     730
Name: count, dtype: int64

# 6. 게임당 샘플링

In [ ]:
def sample_one_game(group, reviews_per_game, mode, random_state):
    """
    게임 1개에 대해 리뷰를 샘플링하는 함수

    mode:
    - recent: 최신 리뷰 우선
    - random: 무작위
    - balanced_by_steam_label: 긍정/부정 리뷰를 가능하면 균형 있게 섞음
    """
    if reviews_per_game is None or len(group) <= reviews_per_game:
        return group

    if mode == "recent":
        return group.sort_values("review_datetime", ascending=False).head(reviews_per_game)

    if mode == "random":
        return group.sample(n=reviews_per_game, random_state=random_state)

    if mode == "balanced_by_steam_label":
        half = reviews_per_game // 2

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        pos_n = min(len(positive), half)
        neg_n = min(len(negative), reviews_per_game - pos_n)

        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)
        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)

        sampled = pd.concat([pos_sample, neg_sample], ignore_index=False)

        # 한쪽 라벨이 부족해서 목표 개수보다 적게 뽑힌 경우 나머지를 전체에서 추가로 뽑음
        remain_n = reviews_per_game - len(sampled)

        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                remain_sample = remain_pool.sample(
                    n=min(remain_n, len(remain_pool)),
                    random_state=random_state
                )
                sampled = pd.concat([sampled, remain_sample], ignore_index=False)

        return sampled.sample(frac=1, random_state=random_state)

    # mode 값이 잘못 들어오면 안전하게 최신순으로 처리
    return group.sort_values("review_datetime", ascending=False).head(reviews_per_game)


def sample_reviews_per_game(df, reviews_per_game=20, mode="balanced_by_steam_label", random_state=42):
    """
    전체 필터링 데이터에서 게임별로 리뷰를 일정 개수만 샘플링하는 함수
    """
    if len(df) == 0:
        return df.copy().reset_index(drop=True)

    # 수정: pandas 버전에 따라 groupby.apply 결과에서 appid가 빠질 수 있으므로
    # 그룹 키를 명시적으로 다시 넣어 샘플링 결과의 appid 컬럼을 보존한다.
    def sample_group(group):
        group_appid = getattr(group, "name", None)
        group = group.copy()
        if "appid" not in group.columns:
            group["appid"] = group_appid

        sampled_group = sample_one_game(
            group=group,
            reviews_per_game=reviews_per_game,
            mode=mode,
            random_state=random_state,
        )

        if "appid" not in sampled_group.columns:
            sampled_group = sampled_group.copy()
            sampled_group["appid"] = group_appid

        return sampled_group

    try:
        sampled = (
            df.groupby("appid", group_keys=False)
            .apply(sample_group, include_groups=False)
            .reset_index(drop=True)
        )
    except TypeError:
        sampled = (
            df.groupby("appid", group_keys=False)
            .apply(sample_group)
            .reset_index(drop=True)
        )

    return sampled


In [ ]:
df_sampled = sample_reviews_per_game(
    df=df_filtered,
    reviews_per_game=REVIEWS_PER_GAME,
    mode=SAMPLE_MODE,
    random_state=RANDOM_STATE,
)

# 전체 리뷰 수 상한 적용
if MAX_TOTAL_REVIEWS is not None and len(df_sampled) > MAX_TOTAL_REVIEWS:
    df_sampled = df_sampled.sample(
        n=MAX_TOTAL_REVIEWS,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

# 테스트 실행용 개수 제한
if TEST_N is not None and len(df_sampled) > TEST_N:
    if SAMPLE_MODE == "recent":
        df_for_llm = (
            df_sampled
            .sort_values("review_datetime", ascending=False)
            .head(TEST_N)
            .reset_index(drop=True)
        )
    else:
        df_for_llm = (
            df_sampled
            .sample(n=TEST_N, random_state=RANDOM_STATE)
            .reset_index(drop=True)
        )
else:
    df_for_llm = df_sampled.reset_index(drop=True)

print("샘플링 후 리뷰 수:", len(df_sampled))
print("샘플링 후 게임 수:", df_sampled["appid"].nunique())
print("최종 LLM 분석 대상 리뷰 수:", len(df_for_llm))
print("최종 LLM 분석 대상 게임 수:", df_for_llm["appid"].nunique())


샘플링 후 리뷰 수: 142
샘플링 후 게임 수: 17
최종 LLM 분석 대상 리뷰 수: 142
최종 LLM 분석 대상 게임 수: 17


## 6-1. 확인용: 최종 LLM 분석 대상 확인
확인용, 삭제해도 작동

In [ ]:
if RUN_CHECK_CELLS:
    display(
        df_for_llm[
            [
                "recommendationid", "appid", "game_name", "language",
                "steam_label_text", "review_datetime", "release_period",
                "playtime_at_review_hours", "playtime_stage", "genres_text",
                "top_steam_tags_text", "review_text_for_llm"
            ]
        ].head(10)
    )

    display(
        df_for_llm.groupby(["game_name", "steam_label_text"])
        .size()
        .reset_index(name="count")
        .sort_values(["game_name", "steam_label_text"])
    )

,recommendationid,appid,game_name,language,steam_label_text,review_datetime,release_period,playtime_at_review_hours,playtime_stage,genres_text,top_steam_tags_text,review_text_for_llm
0,185826123,1443620,Pirates VR: Jolly Roger,english,negative,2025-01-18 02:34:27,D0-D7,1.050000,early,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Just not very fun trying to find little items in the dark. Parrot is extremely annoying. No idea who thought it was ...
1,185700235,1443620,Pirates VR: Jolly Roger,english,negative,2025-01-16 09:48:03,D0-D7,0.650000,early,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Feels like a low quality PCVR game with nothing really new or unique. Bad gameplay and really poor animations. Had t...
2,185757700,1443620,Pirates VR: Jolly Roger,english,positive,2025-01-17 03:51:20,D0-D7,1.700000,early,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...","Not the most user-friendly controls, but I like the idea and implementation"
3,186154119,1443620,Pirates VR: Jolly Roger,english,negative,2025-01-22 13:49:32,D8-D30,0.183333,very_early,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Weird swimming mechanics\nAnnoying parrot\nClimbing Sim\nNot manny VR motions (like bag interaction etc.)\n
4,185713971,1443620,Pirates VR: Jolly Roger,english,positive,2025-01-16 14:43:45,D0-D7,3.033333,mid,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...","It would probably be one of the best games in 2017, but for 2025 it's just a good game that's unlikely to surprise a..."
5,185590398,1443620,Pirates VR: Jolly Roger,english,positive,2025-01-14 17:07:11,D0-D7,0.266667,very_early,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...","fun so far , runs good :) which is the most important"
6,185681266,1443620,Pirates VR: Jolly Roger,english,positive,2025-01-16 01:01:46,D0-D7,2.833333,mid,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Website Review: https://gameordie.net/pirates-vr-jolly-roger-2025/\nYouTube Video Review: https://youtu.be/PLZnl4afK...
7,185712708,1443620,Pirates VR: Jolly Roger,english,negative,2025-01-16 14:22:19,D0-D7,2.283333,mid,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",This wasn't the game that was at all represented in the store trailer or images. It's an escape room and climbing si...
8,185798787,1443620,Pirates VR: Jolly Roger,english,positive,2025-01-17 18:12:35,D0-D7,4.250000,mid,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...","Davy Jones, now you know the real winning pirate!\n(it was scary when he threw coins at me)\nwe did it, along with t..."
9,185638063,1443620,Pirates VR: Jolly Roger,english,negative,2025-01-15 11:00:19,D0-D7,0.416667,very_early,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...","I genuinely wanted to love this game, and I really looked forward to it being released. But after playing it, it fee..."


,game_name,steam_label_text,count
0,BOBR SPRINT,positive,2
1,Bachelairs,positive,9
2,Bureau of Contacts,negative,4
3,Bureau of Contacts,positive,8
4,Dark Shadows,positive,1
5,Detective Penguin,negative,1
6,Detective Penguin,positive,11
7,Dojo Masters,negative,3
8,Dojo Masters,positive,9
9,Echo Point Nova,negative,6


# 7. LLM 출력 스키마 정의

- 결과 컬럼이 매번 달라지는 것을 방지한다.
- JSON 파싱 오류를 줄인다.
- 후속 집계표를 안정적으로 만들 수 있다.

| 설정값                       | 의미          |
| -------------------------- | ----------- |
| `bug`                      | 버그          |
| `optimization`             | 최적화         |
| `performance`              | 성능/프레임      |
| `crash`                    | 튕김/실행 불가    |
| `control`                  | 조작감         |
| `balance`                  | 밸런스         |
| `difficulty`               | 난이도         |
| `content_volume`           | 콘텐츠 양       |
| `story`                    | 스토리         |
| `translation_localization` | 번역/현지화      |
| `ui_ux`                    | UI/UX       |
| `price_value`              | 가격 대비 가치    |
| `multiplayer_network`      | 멀티/서버       |
| `save_progression`         | 저장/진행도      |
| `graphics_audio`           | 그래픽/사운드     |
| `gameplay_loop`            | 핵심 재미/반복 구조 |
| `monetization`             | 과금/DLC      |
| `developer_communication`  | 개발자 소통      |
| `positive_praise`          | 전반적 칭찬      |
| `progression_grind`        | 성장/노가다      |
| `other`                    | 기타          |


In [ ]:
# ============================================================
# 7. LLM 출력 스키마 정의
# ============================================================
# 중요:
# 아래의  category 목록은 이후 집계/시각화의 기준이 되기 때문에
# category 이름을 바꾸면 뒤쪽 groupby 결과도 달라질 수 있다.
# 되도록 한 번 정한 분류 체계는 프로젝트 중간에 자주 바꾸지 않는 것이 좋다.


# ------------------------------------------------------------
class IssueTag(BaseModel):
    category: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰에서 언급된 세부 이슈 카테고리")

    # sentiment는 해당 이슈에 대한 감정 방향
    # 리뷰 전체 감정이 아닌, 이 세부 이슈에 대한 감정
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="이 이슈에 대한 감정"
    )

    # evidence는 왜 이 카테고리/감정으로 판단했는지에 대한 짧은 근거
    # 원문 리뷰를 바탕으로 작성
    evidence: str = Field(
        description="원문 리뷰를 바탕으로 한 짧은 판단 근거",
        min_length=1,
        max_length=160
    )


class SteamReviewAnalysis(BaseModel):
    # 입력 리뷰 ID를 그대로 반환
    # 나중에 원본 리뷰와 LLM 결과를 정확히 매칭하기 위한 핵심 키
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    # LLM이 리뷰 본문만 보고 판단한 감정
    # Steam의 voted_up과 다를 수 있다
    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 감정"
    )

    # 감정을 1~5점으로 수치화한 값
    # 1은 매우 부정, 3은 중립, 5는 매우 긍정으로 해석
    sentiment_score: int = Field(
        ge=1,
        le=5,
        description="1=매우 부정, 3=중립, 5=매우 긍정"
    )

    # 리뷰에서 가장 중요하다고 판단되는 핵심 이슈 1개입니다.
    primary_issue: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰의 대표 이슈")

    # 리뷰 안에서 발견된 여러 세부 이슈
    issue_tags: List[IssueTag] = Field(
        default_factory=list,
        description="리뷰에서 발견된 세부 이슈 목록"
    )

    # 개발사 입장에서 대응 긴급도
    urgency: Literal["low", "medium", "high"] = Field(
        description="개선 필요도 또는 대응 긴급도"
    )

    # 리뷰 요약
    summary: str = Field(
        description="리뷰 핵심 내용 요약",
        min_length=5,
        max_length=220
    )

    # 개발사 관점의 후속 액션
    suggested_action: str = Field(
        description="개발사 또는 운영자가 참고할 수 있는 개선 방향",
        min_length=5,
        max_length=260
    )

class BatchSteamReviewAnalysis(BaseModel):
    """한 번의 배치 요청에서 여러 리뷰 결과를 받을 수 있도록 감싸는 모델."""
    results: List[SteamReviewAnalysis] = Field(
        description="리뷰별 분석 결과 목록"
    )

# 8. Agent 생성

| 설정값            | 의미            |
| -------------- | ------------- |
| `GEMINI_MODEL` | 사용할 Gemini 모델 |
| `temperature`  | 답변의 랜덤성/창의성   |


In [ ]:
# LLM에게 부여할 시스템 프롬프트
system_prompt = """
당신은 Steam 인디게임 리뷰 분석 전문가입니다.

각 리뷰에 대해 다음을 판단하세요.
1. 리뷰 본문 기준 감정(llm_sentiment)
2. 가장 핵심적인 이슈(primary_issue)
3. 세부 이슈(issue_tags)
4. 개발사 관점 suggested_action
5. 개선 필요도 urgency

중요 규칙:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up은 참고 정보일 뿐, 감정은 review 텍스트 기준으로 판단하세요.
- 추천 리뷰라도 불만이 많으면 mixed 또는 negative로 판단할 수 있습니다.
- 비추천 리뷰라도 장단점이 섞여 있으면 mixed로 판단할 수 있습니다.
- issue_tags에는 실제로 언급된 것만 넣으세요.
- evidence는 리뷰 본문에 근거가 있는 짧은 표현 또는 요약으로 작성하세요.
- review가 매우 짧거나 밈/농담 위주면 과잉 해석하지 마세요.
- 게임 메타데이터와 태그는 맥락 참고용이며, 리뷰 본문에 없는 내용을 억지로 추론하지 마세요.
- 개발사 개선 제안은 인디게임 개발사가 실제로 참고할 수 있게 구체적으로 작성하세요.
"""

# Gemini 모델 세부 설정
review_settings = GoogleModelSettings(
    temperature=0.0 # 답변의 랜덤성/창의성
)

# API 키가 있을 때만 Agent를 생성한다.
# RUN_LLM=False로 후속 요약표만 만드는 경우에는 Agent가 없어도 된다.
if api_key:
    review_agent = Agent(
        model_id,
        output_type=BatchSteamReviewAnalysis,
        system_prompt=system_prompt,
        retries=3,
        output_retries=3,
    )
else:
    review_agent = None

print("Agent 생성 여부:", "O" if review_agent is not None else "X")

Agent 생성 여부: O


# 9. 프롬프트 생성

In [ ]:
def build_batch_prompt(batch_df):
    """
    여러 개의 리뷰를 한 번에 LLM에게 보내기 위한 프롬프트 생성 함수.

    입력:
    - batch_df: LLM에게 보낼 리뷰 DataFrame

    출력:
    - prompt: 리뷰 여러 개가 포함된 문자열
    """
    blocks = []

    for _, row in batch_df.iterrows():
        block = f"""
[REVIEW]
recommendationid: {row["recommendationid"]}
appid: {row["appid"]}
game_name: {row.get("game_name", "")}
genres: {row.get("genres_text", "")}
categories: {row.get("categories_text", "")}
steam_top_tags: {row.get("top_steam_tags_text", "")}
stratum: {row.get("stratum", "")}
release_date: {row.get("release_date", "")}
review_datetime: {row.get("review_datetime", "")}
days_from_release: {row.get("days_from_release", "")}
release_period: {row.get("release_period", "")}
language: {row.get("language", "")}
voted_up: {row.get("voted_up", "")}
steam_label_text: {row.get("steam_label_text", "")}
playtime_at_review_hours: {row.get("playtime_at_review_hours", "")}
playtime_stage: {row.get("playtime_stage", "")}
votes_up: {row.get("votes_up", "")}
weighted_vote_score: {row.get("weighted_vote_score", "")}
received_for_free: {row.get("received_for_free", "")}
written_during_early_access: {row.get("written_during_early_access", "")}

review:
{row.get("review_text_for_llm", "")}
[/REVIEW]
"""
        blocks.append(block)

    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\n"
        "결과는 지정된 Pydantic 스키마에 맞게 반환하세요.\n\n"
        + "\n".join(blocks)
    )

    return prompt

# 10. 비동기 분석

In [ ]:
# 동시에 실행될 LLM 요청 수를 제한하기 위한 Semaphore
# MAX_CONCURRENT=1이면 한 번에 요청 1개만 실행한다.
sem = asyncio.Semaphore(MAX_CONCURRENT)


def extract_usage_tokens(result):
    """
    Pydantic AI 결과 객체에서 토큰 사용량을 안전하게 추출하는 함수.

    버전이나 모델에 따라 usage 속성명이 조금 다를 수 있어
    여러 후보 이름을 순서대로 확인한다.
    """
    input_tokens = 0
    output_tokens = 0

    try:
        usage = result.usage()

        input_tokens = (
            getattr(usage, "input_tokens", None)
            or getattr(usage, "request_tokens", None)
            or getattr(usage, "prompt_tokens", None)
            or 0
        )

        output_tokens = (
            getattr(usage, "output_tokens", None)
            or getattr(usage, "response_tokens", None)
            or getattr(usage, "completion_tokens", None)
            or 0
        )

    except Exception:
        pass

    return int(input_tokens or 0), int(output_tokens or 0)

In [ ]:
async def analyze_batch(batch_df, all_results, stats, pbar):
    """
    리뷰 배치 1개를 LLM으로 분석하는 비동기 함수.

    핵심 흐름:
    1. batch_df를 LLM 프롬프트로 변환
    2. LLM 호출
    3. Pydantic 구조로 받은 결과를 원본 리뷰와 매칭
    4. 결과를 all_results에 추가
    5. 실패하면 재시도하고, 최종 실패 시 실패 기록을 남김
    """
    async with sem:
        prompt = build_batch_prompt(batch_df)

        for attempt in range(MAX_RETRIES):
            try:
                if review_agent is None:
                    raise RuntimeError("review_agent가 생성되지 않았습니다. GEMINI_API_KEY와 pydantic_ai 설치 여부를 확인하세요.")

                # LLM Agent 실행
                if review_settings is not None:
                    result = await review_agent.run(prompt, model_settings=review_settings)
                else:
                    result = await review_agent.run(prompt)

                output = result.output

                input_tokens, output_tokens = extract_usage_tokens(result)
                stats["input_tokens"] += input_tokens
                stats["output_tokens"] += output_tokens
                stats["requests"] += 1

                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                for item in output.results:
                    rid = str(item.recommendationid)

                    # LLM이 입력에 없던 ID를 반환하면 무시한다.
                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)

                    record = {
                        # 중요
                        # "primary_issue": item.primary_issue : 부정/긍정 반응의 핵심 원인 분류
                        # "urgency": item.urgency, : 개선 우선순위 판단
                        # "suggested_action": item.suggested_action, : 패치/운영 방향 제안
                        "analysis_status": "success",

                        # 원본 리뷰 식별 정보
                        "recommendationid": rid,                         # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
                        "appid": row["appid"],                           # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
                        "game_name": row.get("game_name", ""),           # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.
                        "language": row.get("language", ""),             # 리뷰 작성 언어입니다. 영어/한국어 등 언어별 분석이나 필터링에 사용합니다.
                        "review_datetime": row.get("review_datetime", None),      # 리뷰 작성 일시입니다. 시계열 분석이나 출시 후 경과일 계산에 사용합니다.
                        "timestamp_created": row.get("timestamp_created", None),  # Steam 원본 리뷰 작성 Unix timestamp입니다. 원본 시간값 보존용입니다.

                        # Steam 원본 라벨/메타
                        "steam_voted_up": bool(row.get("voted_up", False)),       # Steam 기준 추천 여부입니다. True면 긍정 리뷰, False면 부정 리뷰입니다.
                        "steam_label_text": row.get("steam_label_text", ""),      # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),  # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
                        "playtime_forever_hours": row.get("playtime_forever_hours", None),      # 유저의 전체 누적 플레이타임입니다. 리뷰 작성 이후 플레이 지속 여부를 참고할 수 있습니다.
                        "playtime_stage": row.get("playtime_stage", None),        # 플레이타임을 구간화한 값입니다. 초반/중반/장기 플레이 리뷰를 구분할 때 사용합니다.
                        "votes_up": row.get("votes_up", None),                    # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
                        "votes_funny": row.get("votes_funny", None),              # 해당 리뷰가 받은 '재미있음' 투표 수입니다. 밈성/농담성 리뷰 참고용입니다.
                        "weighted_vote_score": row.get("weighted_vote_score", None),  # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.
                        "comment_count": row.get("comment_count", None),          # 리뷰에 달린 댓글 수입니다. 논쟁성이나 관심도 참고용입니다.
                        "steam_purchase": row.get("steam_purchase", None),        # Steam에서 직접 구매한 유저의 리뷰인지 여부입니다. 리뷰 신뢰도 필터링에 사용합니다.
                        "received_for_free": row.get("received_for_free", None),  # 무료로 받은 게임인지 여부입니다. 일반 구매자 반응과 구분할 때 사용합니다.
                        "written_during_early_access": row.get("written_during_early_access", None),  # 얼리액세스 기간에 작성된 리뷰인지 여부입니다.

                        # 게임 메타
                        "release_date": row.get("release_date", None),            # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
                        "days_from_release": row.get("days_from_release", None),  # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
                        "release_period": row.get("release_period", None),        # 출시 후 기간 구간입니다. 예: pre_release, D0-D7, D8-D30, D31-D90 등입니다.
                        "genres_text": row.get("genres_text", ""),                # 게임 장르 목록을 문자열로 정리한 값입니다. 장르별 반응 분석에 사용합니다.
                        "categories_text": row.get("categories_text", ""),        # 게임 카테고리/플레이 방식 목록입니다. 싱글/멀티/협동 여부 분석에 사용합니다.
                        "top_steam_tags_text": row.get("top_steam_tags_text", ""), # 주요 Steam 태그 목록입니다. 태그 기반 유사 게임 분석이나 반응 비교에 사용합니다.
                        "price": row.get("price", None),                          # 게임 가격입니다. 가격대별 초기 반응 분석에 사용합니다.
                        "price_group": row.get("price_group", None),              # 가격을 구간화한 값입니다. 예: free, 0-5, 5-10, 10-20 등입니다.
                        "stratum": row.get("stratum", None),                      # 표본 추출 시 사용한 층화 그룹입니다. 흥행 규모/리뷰 신뢰도 기준 분석에 사용합니다.

                        # LLM 분석 결과
                        "llm_sentiment": item.llm_sentiment,                      # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
                        "sentiment_score": item.sentiment_score,                  # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
                        "primary_issue": item.primary_issue,                      # LLM이 판단한 리뷰의 대표 이슈입니다. 버그/밸런스/콘텐츠/가격 등 주요 원인을 나타냅니다.
                        "issue_tags": [tag.model_dump() for tag in item.issue_tags],  # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
                        "urgency": item.urgency,                                  # 개선 시급도입니다. 어떤 문제를 먼저 고쳐야 할지 우선순위 판단에 사용합니다.
                        "summary": item.summary,                                  # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
                        "suggested_action": item.suggested_action,                # LLM이 제안한 개선 방향입니다. 패치/운영 방향 제안에 활용합니다.
                    }

                    all_results.append(to_serializable(record))

                # LLM 결과에서 누락된 ID 기록
                missing_ids = input_ids - matched_ids
                for missing_id in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == missing_id].iloc[0]
                    all_results.append(to_serializable({
                        "analysis_status": "missing_in_llm_output",
                        "recommendationid": missing_id,
                        "appid": row["appid"],
                        "game_name": row.get("game_name", ""),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "review_datetime": row.get("review_datetime", None),
                    }))

                save_checkpoint(all_results)
                pbar.update(len(batch_df))
                return

            except Exception as e:
                wait_sec = 2 ** attempt
                print(f"배치 분석 실패: attempt={attempt + 1}/{MAX_RETRIES}, error={e}")
                time.sleep(wait_sec)

        # 최대 재시도 후에도 실패한 경우 실패 기록 저장
        for _, row in batch_df.iterrows():
            all_results.append(to_serializable({
                "analysis_status": "failed",
                "recommendationid": str(row["recommendationid"]),
                "appid": row["appid"],
                "game_name": row.get("game_name", ""),
                "steam_label_text": row.get("steam_label_text", ""),
                "review_datetime": row.get("review_datetime", None),
            }))

        save_checkpoint(all_results)
        pbar.update(len(batch_df))

In [ ]:
async def run_analysis(df):
    """
    전체 LLM 분석을 실행하는 함수.

    처리 흐름:
    1. 기존 checkpoint를 읽는다.
    2. 이미 처리된 recommendationid는 제외한다.
    3. 남은 리뷰를 BATCH_SIZE 단위로 나눈다.
    4. CHUNK_SIZE 단위로 비동기 요청을 실행한다.
    5. 중간 결과는 checkpoint에 계속 저장한다.
    """
    if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()
        print("기존 checkpoint 삭제:", CHECKPOINT_PATH)

    all_results = load_checkpoint()
    all_results = list(all_results)

    # 현재 분석 대상 df에 포함된 ID만 checkpoint에서 유지한다.
    target_ids = set(df["recommendationid"].astype(str))
    all_results = [
        row for row in all_results
        if str(row.get("recommendationid")) in target_ids
    ]

    done_ids = {
        str(row.get("recommendationid"))
        for row in all_results
        if row.get("analysis_status") in ["success", "missing_in_llm_output", "failed"]
    }

    to_process = df[~df["recommendationid"].astype(str).isin(done_ids)].copy()

    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(done_ids),
        "to_process_count": len(to_process),
    }

    if len(to_process) == 0:
        print("새로 처리할 리뷰가 없습니다. checkpoint 또는 기존 결과를 사용합니다.")
        return all_results, stats

    batches = [
        to_process.iloc[i:i + BATCH_SIZE]
        for i in range(0, len(to_process), BATCH_SIZE)
    ]

    with tqdm(total=len(to_process), desc="LLM 리뷰 분석 진행") as pbar:
        for start in range(0, len(batches), CHUNK_SIZE):
            chunk = batches[start:start + CHUNK_SIZE]

            tasks = [
                analyze_batch(batch_df, all_results, stats, pbar)
                for batch_df in chunk
            ]

            await asyncio.gather(*tasks)

            if REQUEST_SLEEP_SEC > 0:
                await asyncio.sleep(REQUEST_SLEEP_SEC)

    return all_results, stats

# 11. 실행

In [ ]:
# 전체 LLM 분석 실행

# RUN_LLM=True이면 실제 Gemini API를 호출
# RUN_LLM=False이면 기존 RESULT_JSON 또는 CHECKPOINT만 읽어서 후속 요약표 생성

if RUN_LLM:
    results, stats = await run_analysis(df_for_llm)
else:
    results = load_existing_results()
    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(results),
        "to_process_count": 0,
    }

print_cost_report(
    input_tokens=stats["input_tokens"],
    output_tokens=stats["output_tokens"],
    requests=stats["requests"],
    checkpoint_count=stats["checkpoint_count"],
    to_process_count=stats["to_process_count"],
)

# 결과 저장 전, numpy/pandas 타입을 JSON 저장 가능한 타입으로 변환
safe_results = to_serializable(results)

# 최종 분석 결과를 pandas DataFrame으로 변환
df_result = pd.DataFrame(safe_results)

print("분석 결과 행 수:", len(df_result))

기존 checkpoint 삭제: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\steam_review_llm_checkpoint.json


LLM 리뷰 분석 진행: 100%|██████████| 142/142 [11:50<00:00,  5.00s/it]

토큰 사용량 / 예상 비용
요청 수: 48
checkpoint에서 불러온 리뷰 수: 0
이번 실행에서 새로 처리할 리뷰 수: 142
입력 토큰: 145,036
출력 토큰: 139,597
예상 비용(USD): $0.106057
예상 비용(KRW): ₩159
분석 결과 행 수: 142


# 12. 결과 후처리 및 저장

In [ ]:
def classify_sentiment_relation(row):
    """Steam 라벨과 LLM 감정의 관계 분류"""
    steam_label = row.get("steam_label_text")
    llm_sentiment = row.get("llm_sentiment")

    if steam_label == "positive":
        if llm_sentiment == "positive":
            return "exact_match"
        elif llm_sentiment == "mixed":
            return "partial_match"
        elif llm_sentiment == "negative":
            return "mismatch"
        else:
            return "unclear"

    elif steam_label == "negative":
        if llm_sentiment == "negative":
            return "exact_match"
        elif llm_sentiment == "mixed":
            return "partial_match"
        elif llm_sentiment == "positive":
            return "mismatch"
        else:
            return "unclear"

    else:
        return "unknown"


# df_result가 비어있지 않은 경우에만 후처리 및 저장
if len(df_result) > 0:
    # 날짜형 문자열이 들어왔을 수 있으므로 다시 datetime으로 정리
    if "review_datetime" in df_result.columns:
        df_result["review_datetime"] = pd.to_datetime(df_result["review_datetime"], errors="coerce")

    if "release_date" in df_result.columns:
        df_result["release_date"] = pd.to_datetime(df_result["release_date"], errors="coerce")

    # Steam 라벨과 LLM 감정 관계 컬럼 생성
    df_result["sentiment_relation"] = df_result.apply(classify_sentiment_relation, axis=1)

    # JSON/CSV 저장
    safe_results = to_serializable(df_result.to_dict(orient="records"))

    with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)

    df_result.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8-sig")

    print("JSON 저장:", RESULT_JSON_PATH)
    print("CSV 저장:", RESULT_CSV_PATH)
    print("분석 리뷰 수:", len(df_result))

else:
    print("분석 결과가 없습니다.")
    print("실제 분석을 하려면 RUN_LLM=True로 설정하고 실행하세요.")
    print("또는 기존 RESULT_JSON/CHECKPOINT 파일이 있는지 확인하세요.")

JSON 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\steam_review_llm_results.json
CSV 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\steam_review_llm_results.csv
분석 리뷰 수: 142


# 13. 게임 단위 종합 요약 및 개선 제안

`df_result`와 `df_tags`를 기반으로 게임 단위 결과를 만든다.

- `df_result`: 리뷰 1건당 LLM 분석 결과가 들어 있는 메인 테이블
- `df_tags`: `df_result["issue_tags"]`를 펼쳐 만든 세부 이슈 태그 테이블
- 최종 목표: 리뷰별 결과를 그대로 보여주는 것이 아니라, 게임별 종합 반응과 개선 방향을 요약


In [ ]:
## 2. groupby 집계에 사용할 함수
def get_most_common_value(series):
    """
    그룹 안에서 가장 많이 등장한 값을 반환

    예:
    - 한 게임의 llm_sentiment가 positive 4개, mixed 1개라면 positive 반환
    - 값이 모두 비어 있으면 None 반환
    """
    cleaned = series.dropna()

    if len(cleaned) == 0:
        return None

    return cleaned.value_counts().idxmax()


def count_high_urgency(series):
    """urgency 값 중 high 개수 계산"""
    return (series == "high").sum()


def count_medium_urgency(series):
    """urgency 값 중 medium 개수 계산"""
    return (series == "medium").sum()


def count_low_urgency(series):
    """urgency 값 중 low 개수 계산"""
    return (series == "low").sum()

In [ ]:
## 3. 출력용 문자열 생성 함수
def format_count_items(count_series, mapping=None, top_n=3):
    """
    value_counts 결과를 보고서용 문자열로 바꾸는 함수

    예:
    bug 3, optimization 2 -> 버그(3), 최적화/성능(2)
    """
    if count_series is None or len(count_series) == 0:
        return "해당 없음"

    items = []

    for key, count in count_series.head(top_n).items():
        label = mapping.get(key, key) if mapping else key
        items.append(f"{label}({count})")

    return ", ".join(items)

## 14. 게임 단위 종합표 생성

### 14-1. 게임별 기본 지표 집계

In [ ]:
ISSUE_KR_MAP = {
    "bug": "버그",
    "optimization": "최적화",
    "performance": "성능",
    "crash": "크래시",
    "control": "조작감",
    "balance": "밸런스",
    "difficulty": "난이도",
    "content_volume": "콘텐츠 분량",
    "story": "스토리",
    "translation_localization": "번역/현지화",
    "ui_ux": "UI/UX",
    "price_value": "가격/가치",
    "multiplayer_network": "멀티/네트워크",
    "save_progression": "저장/진행",
    "graphics_audio": "그래픽/사운드",
    "gameplay_loop": "게임플레이 루프",
    "monetization": "과금",
    "developer_communication": "개발사 소통",
    "positive_praise": "긍정 칭찬",
    "short_playtime_rejection": "짧은 플레이 후 이탈",
    "progression_grind": "성장/반복 노가다",
    "other": "기타",
}

In [ ]:
# df_result는 리뷰 1건당 1행인 LLM 분석 결과 테이블

if len(df_result) > 0 and "analysis_status" in df_result.columns:
    df_success = df_result[df_result["analysis_status"] == "success"].copy()
else:
    df_success = pd.DataFrame()

if len(df_success) > 0:
    game_summary = (
        df_success
        .groupby(["appid", "game_name"], as_index=False)
        .agg(
            analyzed_review_count=("recommendationid", "count"),
            steam_positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
            steam_negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
            main_llm_sentiment=("llm_sentiment", get_most_common_value),
            main_issue=("primary_issue", get_most_common_value),
            high_urgency_count=("urgency", count_high_urgency),
            medium_urgency_count=("urgency", count_medium_urgency),
            low_urgency_count=("urgency", count_low_urgency),
            release_date=("release_date", "first"),
            genres=("genres_text", "first"),
            categories=("categories_text", "first"),
            top_tags=("top_steam_tags_text", "first"),
            stratum=("stratum", "first"),
            price_group=("price_group", "first"),
        )
    )

    # 게임별 주요 이슈 top3 문자열 생성
    issue_text_rows = []
    for appid, group in df_success.groupby("appid"):
        counts = group["primary_issue"].value_counts()
        issue_text_rows.append({
            "appid": appid,
            "top_issues": format_count_items(counts, mapping=ISSUE_KR_MAP, top_n=3)
        })

    issue_text_df = pd.DataFrame(issue_text_rows)
    game_summary = game_summary.merge(issue_text_df, on="appid", how="left")

    game_summary.to_csv(GAME_SUMMARY_PATH, index=False, encoding="utf-8-sig")

    print("게임 단위 요약 저장:", GAME_SUMMARY_PATH)
    display(game_summary.head(20))

else:
    game_summary = pd.DataFrame()
    print("성공한 LLM 분석 결과가 없어 게임 단위 요약표를 만들지 않았습니다.")

게임 단위 요약 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\llm_game_summary.csv


,appid,game_name,analyzed_review_count,steam_positive_count,steam_negative_count,main_llm_sentiment,main_issue,high_urgency_count,medium_urgency_count,low_urgency_count,release_date,genres,categories,top_tags,stratum,price_group,top_issues
0,1443620,Pirates VR: Jolly Roger,12,6,6,negative,gameplay_loop,5,4,3,2025-01-14,"Action, Adventure, Indie","Single-player, Steam Achievements, Tracked Controller Support, VR Only, Family Sharing","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Action_mid,10-20,"게임플레이 루프(4), 긍정 칭찬(3), 조작감(2)"
1,1836730,Echo Point Nova,12,6,6,positive,positive_praise,5,1,6,2024-09-24,"Action, Adventure, Indie","Single-player, Multi-player, Co-op, Online Co-op, Steam Achievements, Full controller support, Steam Workshop, Steam...","FPS, Shooter, Online Co-Op, Exploration, Action, Arena Shooter, First-Person, 3D, Stylized, Adventure",Action_high,10-20,"긍정 칭찬(6), 게임플레이 루프(4), 버그(1)"
2,2008980,XENOTILT: HOSTILE PINBALL ACTION,12,10,2,positive,positive_praise,2,2,8,2024-11-12,"Action, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Stats, Steam Leaderboards, Family Sharing","Bullet Hell, Shoot 'Em Up, Difficult, Hack and Slash, Physics, Unforgiving, Arcade, 2D, Top-Down, Cyberpunk",Action_high,10-20,"긍정 칭찬(9), 기타(1), UI/UX(1)"
3,2193490,First Cut: Samurai Duel,12,6,6,positive,positive_praise,6,2,4,2024-01-17,"Action, Indie","Single-player, Multi-player, PvP, Shared/Split Screen PvP, Shared/Split Screen, Steam Achievements, Full controller ...","Side Scroller, Hack and Slash, Martial Arts, 2D Fighter, PvP, PvE, Arcade, Swordplay, Character Customization, 2D",Action_high,5-10,"긍정 칭찬(4), 난이도(3), 멀티/네트워크(2)"
4,2467000,BOBR SPRINT,2,2,0,mixed,control,1,0,1,2024-06-15,"Action, Adventure, Indie","Single-player, Full controller support, Family Sharing","2D Platformer, Adventure, Action-Adventure, Precision Platformer, Platformer, Pixel Graphics, Old School, 1990's, Du...",Action_mid,0-5,"조작감(1), 긍정 칭찬(1)"
5,2538000,Dojo Masters,12,9,3,positive,positive_praise,3,3,6,2024-07-17,"Action, Indie","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Co-op, Online Co-op, Shared/Split Screen Co-o...","Action, 2D Fighter, Arcade, 2D, Spectacle fighter, Pixel Graphics, Martial Arts, Retro, Character Customization, Combat",Action_mid,10-20,"긍정 칭찬(4), 게임플레이 루프(3), 콘텐츠 분량(2)"
6,2599630,Zorro and Zedd,12,9,3,positive,positive_praise,3,1,8,2025-04-11,"Action, Adventure, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Family Sharing","Adventure, 3D Platformer, Dragons, Dark Fantasy, Hack and Slash, Parkour, Platformer, Action-Adventure, 3D, Third Pe...",Action_low,5-10,"긍정 칭찬(8), 게임플레이 루프(2), 조작감(1)"
7,2738110,Vengeance Hunters,12,10,2,positive,positive_praise,2,2,8,2024-10-27,"Action, Indie","Single-player, Multi-player, Co-op, Shared/Split Screen Co-op, Shared/Split Screen, Steam Achievements, Full control...","Action, Beat 'em up, Pixel Graphics, Post-apocalyptic, Retro, Combat, Indie, Local Co-Op, Multiplayer, Arcade",Action_mid,10-20,"긍정 칭찬(8), 게임플레이 루프(2), 밸런스(1)"
8,2790180,The Recurrence,4,4,0,positive,positive_praise,0,0,4,2024-02-12,"Action, Adventure, Indie","Single-player, Steam Achievements, Full controller support, Family Sharing","Action, Action-Adventure, Rogue-like, Exploration, FPS, 3D, First-Person, Voxel, Dark, Horror",Action_low,0-5,긍정 칭찬(4)
9,2812480,MoonWhite,2,2,0,mixed,bug,0,1,1,2024-03-30,"Action, Indie","Single-player, Partial Controller Support, Family Sharing","Pixel Graphics, 2D Fighter, Action, Martial Arts, Swordplay, Combat, Nature, Rhythm, Spectacle fighter, Snow",Action_mid,0-5,"버그(1), 긍정 칭찬(1)"


# 15. 이슈 태그 펼치기

In [ ]:
def flatten_issue_tags(df):
    """
    리뷰별 issue_tags 리스트를 행 단위로 펼치는 함수.

    결과:
    - 리뷰 1개에 issue_tags가 3개 있으면 3행으로 변환된다.
    - 태그별 빈도, 감정, 근거를 따로 집계할 때 사용한다.
    """
    flat_rows = []

    if len(df) == 0 or "issue_tags" not in df.columns:
        return pd.DataFrame()

    for _, row in df.iterrows():
        tags = row.get("issue_tags", [])

        if not isinstance(tags, list):
            continue

        for tag in tags:
            if not isinstance(tag, dict):
                continue

            flat_rows.append({
                "recommendationid": row.get("recommendationid"),
                "appid": row.get("appid"),
                "game_name": row.get("game_name"),
                "steam_label_text": row.get("steam_label_text"),
                "llm_sentiment": row.get("llm_sentiment"),
                "primary_issue": row.get("primary_issue"),
                "urgency": row.get("urgency"),
                "release_period": row.get("release_period"),
                "genres_text": row.get("genres_text"),
                "top_steam_tags_text": row.get("top_steam_tags_text"),
                "tag_category": tag.get("category"),
                "tag_sentiment": tag.get("sentiment"),
                "tag_evidence": tag.get("evidence"),
            })

    return pd.DataFrame(flat_rows)


if len(df_success) > 0:
    df_issue_tags_flat = flatten_issue_tags(df_success)
    df_issue_tags_flat.to_csv(ISSUE_TAG_FLAT_PATH, index=False, encoding="utf-8-sig")

    print("이슈 태그 펼친 결과 저장:", ISSUE_TAG_FLAT_PATH)
    print("이슈 태그 행 수:", len(df_issue_tags_flat))
    display(df_issue_tags_flat.head(20))
else:
    df_issue_tags_flat = pd.DataFrame()
    print("성공한 LLM 분석 결과가 없어 이슈 태그 펼치기를 수행하지 않았습니다.")

이슈 태그 펼친 결과 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\llm_issue_tags_flat.csv
이슈 태그 행 수: 463


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,primary_issue,urgency,release_period,genres_text,top_steam_tags_text,tag_category,tag_sentiment,tag_evidence
0,185826123,1443620,Pirates VR: Jolly Roger,negative,negative,gameplay_loop,high,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",gameplay_loop,negative,not very fun trying to find little items in the dark
1,185826123,1443620,Pirates VR: Jolly Roger,negative,negative,gameplay_loop,high,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",ui_ux,negative,Parrot is extremely annoying. No idea who thought it was a good idea to have a character follow you around insulting...
2,185700235,1443620,Pirates VR: Jolly Roger,negative,negative,gameplay_loop,high,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",other,negative,Feels like a low quality PCVR game with nothing really new or unique.
3,185700235,1443620,Pirates VR: Jolly Roger,negative,negative,gameplay_loop,high,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",gameplay_loop,negative,Bad gameplay
4,185700235,1443620,Pirates VR: Jolly Roger,negative,negative,gameplay_loop,high,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",graphics_audio,negative,really poor animations
5,185757700,1443620,Pirates VR: Jolly Roger,positive,mixed,control,medium,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",control,negative,Not the most user-friendly controls
6,185757700,1443620,Pirates VR: Jolly Roger,positive,mixed,control,medium,D0-D7,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",positive_praise,positive,I like the idea and implementation
7,186154119,1443620,Pirates VR: Jolly Roger,negative,negative,control,high,D8-D30,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",control,negative,Weird swimming mechanics
8,186154119,1443620,Pirates VR: Jolly Roger,negative,negative,control,high,D8-D30,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",ui_ux,negative,Annoying parrot
9,186154119,1443620,Pirates VR: Jolly Roger,negative,negative,control,high,D8-D30,"Action, Adventure, Indie","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",gameplay_loop,negative,Climbing Sim


# 16. 확인용 요약표
1. 게임별 종합 요약 및 개선 제안
2. 리뷰별 LLM 분석 결과 미리보기
3. 게임별 LLM 감정 분포
4. 게임별 핵심 이슈 분포
5. 게임별 개선 필요도 분포
6. 게임별 Steam 라벨-LLM 감정 관계
7. 게임별 세부 이슈 태그 분포
8. 세부 이슈 태그별 감정 방향 분포
9. 저장된 결과 파일 경로 확인


In [ ]:
# ============================================================
# 16. 최종 결과 출력
# ============================================================

# ============================================================
# 16-0. 출력용 한글 매핑
# ============================================================
# 영어로 저장된 LLM 결과값을 한글명으로 바꾸기 위한 딕셔너리입니다.

sentiment_kor_map = {
    "positive": "긍정",
    "negative": "부정",
    "neutral": "중립",
    "mixed": "혼합",
    "unclear": "불명확",
}

urgency_kor_map = {
    "low": "낮음",
    "medium": "보통",
    "high": "높음",
}

# classify_sentiment_relation() 함수에서 생성되는 값 기준입니다.
relation_kor_map = {
    "exact_match": "Steam 라벨과 LLM 감정 일치",
    "partial_match": "Steam 라벨과 LLM 감정 부분 일치",
    "mismatch": "Steam 라벨과 LLM 감정 불일치",
    "unclear": "LLM 감정 불명확",
    "unknown": "Steam 라벨 없음",
}

# 앞쪽에서 ISSUE_KR_MAP을 이미 만들었으므로 그 기준을 그대로 사용합니다.
issue_kor_map = ISSUE_KR_MAP

# ============================================================
# 16-1. 출력용 테이블 생성 함수
# ============================================================
def make_crosstab(
    data,
    index_col,
    column_col,
    index_name,
    column_name,
    column_mapping=None
):
    """
    게임별 감정 분포, 게임별 이슈 분포처럼
    행에는 게임명, 열에는 분류값을 두고 개수를 세는 교차표를 만드는 함수입니다.
    """

    if len(data) == 0 or index_col not in data.columns or column_col not in data.columns:
        return pd.DataFrame()

    table = pd.crosstab(
        data[index_col],
        data[column_col]
    )

    table = table.rename_axis(
        index=index_name,
        columns=column_name
    )

    if column_mapping is not None:
        table = table.rename(columns=column_mapping)

    return table


# ============================================================
# 16-2. 출력에 사용할 데이터 준비
# ============================================================
# df_result에는 성공/실패 결과가 함께 들어갈 수 있으므로,
# 최종 출력에서는 성공한 LLM 분석 결과만 사용합니다.
if len(df_result) > 0 and "analysis_status" in df_result.columns:
    df_output = df_result[df_result["analysis_status"] == "success"].copy()
else:
    df_output = pd.DataFrame()

df_tags = df_issue_tags_flat.copy() if "df_issue_tags_flat" in globals() else pd.DataFrame()


# ============================================================
# 16-3. 게임 단위 핵심 출력물
# ============================================================
if len(df_output) == 0:
    display("성공한 LLM 분석 결과가 없어 최종 출력표 생성을 건너뜁니다.")
    display("실제 분석을 하려면 RUN_LLM=True로 설정하거나 기존 RESULT_JSON/CHECKPOINT 파일을 준비하세요.")

else:
    # ------------------------------------------------------------
    # 1. 게임별 종합 요약 및 개선 제안
    # ------------------------------------------------------------
    display("=== 1. 게임별 종합 요약 및 개선 제안 ===")
    display(game_summary)


    # ------------------------------------------------------------
    # 2. 리뷰별 LLM 분석 결과 미리보기
    # ------------------------------------------------------------
    display("=== 2. 리뷰별 LLM 분석 결과 미리보기 ===")

    review_preview_cols = [
        "game_name",
        "steam_label_text",
        "llm_sentiment",
        "sentiment_score",
        "primary_issue",
        "urgency",
        "summary",
        "suggested_action",
    ]

    review_preview_cols = [col for col in review_preview_cols if col in df_output.columns]
    display(df_output[review_preview_cols].head(30))


    # ------------------------------------------------------------
    # 3. 게임별 LLM 감정 분포
    # ------------------------------------------------------------
    display("=== 3. 게임별 LLM 감정 분포 ===")

    sentiment_table = make_crosstab(
        data=df_output,
        index_col="game_name",
        column_col="llm_sentiment",
        index_name="게임명",
        column_name="LLM 감정",
        column_mapping=sentiment_kor_map
    )

    display(sentiment_table)


    # ------------------------------------------------------------
    # 4. 게임별 핵심 이슈 분포
    # ------------------------------------------------------------
    display("=== 4. 게임별 핵심 이슈 분포 ===")

    issue_table = make_crosstab(
        data=df_output,
        index_col="game_name",
        column_col="primary_issue",
        index_name="게임명",
        column_name="핵심 이슈",
        column_mapping=issue_kor_map
    )

    display(issue_table)


    # ------------------------------------------------------------
    # 5. 게임별 개선 필요도 분포
    # ------------------------------------------------------------
    display("=== 5. 게임별 개선 필요도 분포 ===")

    urgency_table = make_crosstab(
        data=df_output,
        index_col="game_name",
        column_col="urgency",
        index_name="게임명",
        column_name="개선 필요도",
        column_mapping=urgency_kor_map
    )

    display(urgency_table)


    # ------------------------------------------------------------
    # 6. 게임별 Steam 라벨-LLM 감정 관계
    # ------------------------------------------------------------
    display("=== 6. 게임별 Steam 라벨-LLM 감정 관계 ===")

    relation_table = make_crosstab(
        data=df_output,
        index_col="game_name",
        column_col="sentiment_relation",
        index_name="게임명",
        column_name="Steam 라벨-LLM 감정 관계",
        column_mapping=relation_kor_map
    )

    display(relation_table)


    # ------------------------------------------------------------
    # 7. 게임별 세부 이슈 태그 분포
    # ------------------------------------------------------------
    display("=== 7. 게임별 세부 이슈 태그 분포 ===")

    tag_table = make_crosstab(
        data=df_tags,
        index_col="game_name",
        column_col="tag_category",
        index_name="게임명",
        column_name="세부 이슈 태그",
        column_mapping=issue_kor_map
    )

    display(tag_table)


    # ------------------------------------------------------------
    # 8. 세부 이슈 태그별 감정 방향 분포
    # ------------------------------------------------------------
    display("=== 8. 세부 이슈 태그별 감정 방향 분포 ===")

    if len(df_tags) > 0 and {"tag_category", "tag_sentiment"}.issubset(df_tags.columns):
        tag_sentiment_table = pd.crosstab(
            df_tags["tag_category"],
            df_tags["tag_sentiment"]
        )

        tag_sentiment_table = tag_sentiment_table.rename_axis(
            index="세부 이슈 태그",
            columns="태그 감정"
        )

        tag_sentiment_table = tag_sentiment_table.rename(
            index=issue_kor_map,
            columns=sentiment_kor_map
        )
    else:
        tag_sentiment_table = pd.DataFrame()

    display(tag_sentiment_table)


# ------------------------------------------------------------
# 9. 저장된 결과 파일 경로 확인
# ------------------------------------------------------------
display("=== 9. 저장된 결과 파일 경로 확인 ===")

result_path_table = pd.DataFrame({
    "결과 파일": [
        "리뷰별 LLM 분석 결과 JSON",
        "리뷰별 LLM 분석 결과 CSV",
        "게임 단위 요약 CSV",
        "세부 이슈 태그 CSV",
    ],
    "경로": [
        RESULT_JSON_PATH,
        RESULT_CSV_PATH,
        GAME_SUMMARY_PATH,
        ISSUE_TAG_FLAT_PATH,
    ]
})

display(result_path_table)


'=== 1. 게임별 종합 요약 및 개선 제안 ==='

,appid,game_name,analyzed_review_count,steam_positive_count,steam_negative_count,main_llm_sentiment,main_issue,high_urgency_count,medium_urgency_count,low_urgency_count,release_date,genres,categories,top_tags,stratum,price_group,top_issues
0,1443620,Pirates VR: Jolly Roger,12,6,6,negative,gameplay_loop,5,4,3,2025-01-14,"Action, Adventure, Indie","Single-player, Steam Achievements, Tracked Controller Support, VR Only, Family Sharing","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Action_mid,10-20,"게임플레이 루프(4), 긍정 칭찬(3), 조작감(2)"
1,1836730,Echo Point Nova,12,6,6,positive,positive_praise,5,1,6,2024-09-24,"Action, Adventure, Indie","Single-player, Multi-player, Co-op, Online Co-op, Steam Achievements, Full controller support, Steam Workshop, Steam...","FPS, Shooter, Online Co-Op, Exploration, Action, Arena Shooter, First-Person, 3D, Stylized, Adventure",Action_high,10-20,"긍정 칭찬(6), 게임플레이 루프(4), 버그(1)"
2,2008980,XENOTILT: HOSTILE PINBALL ACTION,12,10,2,positive,positive_praise,2,2,8,2024-11-12,"Action, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Stats, Steam Leaderboards, Family Sharing","Bullet Hell, Shoot 'Em Up, Difficult, Hack and Slash, Physics, Unforgiving, Arcade, 2D, Top-Down, Cyberpunk",Action_high,10-20,"긍정 칭찬(9), 기타(1), UI/UX(1)"
3,2193490,First Cut: Samurai Duel,12,6,6,positive,positive_praise,6,2,4,2024-01-17,"Action, Indie","Single-player, Multi-player, PvP, Shared/Split Screen PvP, Shared/Split Screen, Steam Achievements, Full controller ...","Side Scroller, Hack and Slash, Martial Arts, 2D Fighter, PvP, PvE, Arcade, Swordplay, Character Customization, 2D",Action_high,5-10,"긍정 칭찬(4), 난이도(3), 멀티/네트워크(2)"
4,2467000,BOBR SPRINT,2,2,0,mixed,control,1,0,1,2024-06-15,"Action, Adventure, Indie","Single-player, Full controller support, Family Sharing","2D Platformer, Adventure, Action-Adventure, Precision Platformer, Platformer, Pixel Graphics, Old School, 1990's, Du...",Action_mid,0-5,"조작감(1), 긍정 칭찬(1)"
5,2538000,Dojo Masters,12,9,3,positive,positive_praise,3,3,6,2024-07-17,"Action, Indie","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Co-op, Online Co-op, Shared/Split Screen Co-o...","Action, 2D Fighter, Arcade, 2D, Spectacle fighter, Pixel Graphics, Martial Arts, Retro, Character Customization, Combat",Action_mid,10-20,"긍정 칭찬(4), 게임플레이 루프(3), 콘텐츠 분량(2)"
6,2599630,Zorro and Zedd,12,9,3,positive,positive_praise,3,1,8,2025-04-11,"Action, Adventure, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Family Sharing","Adventure, 3D Platformer, Dragons, Dark Fantasy, Hack and Slash, Parkour, Platformer, Action-Adventure, 3D, Third Pe...",Action_low,5-10,"긍정 칭찬(8), 게임플레이 루프(2), 조작감(1)"
7,2738110,Vengeance Hunters,12,10,2,positive,positive_praise,2,2,8,2024-10-27,"Action, Indie","Single-player, Multi-player, Co-op, Shared/Split Screen Co-op, Shared/Split Screen, Steam Achievements, Full control...","Action, Beat 'em up, Pixel Graphics, Post-apocalyptic, Retro, Combat, Indie, Local Co-Op, Multiplayer, Arcade",Action_mid,10-20,"긍정 칭찬(8), 게임플레이 루프(2), 밸런스(1)"
8,2790180,The Recurrence,4,4,0,positive,positive_praise,0,0,4,2024-02-12,"Action, Adventure, Indie","Single-player, Steam Achievements, Full controller support, Family Sharing","Action, Action-Adventure, Rogue-like, Exploration, FPS, 3D, First-Person, Voxel, Dark, Horror",Action_low,0-5,긍정 칭찬(4)
9,2812480,MoonWhite,2,2,0,mixed,bug,0,1,1,2024-03-30,"Action, Indie","Single-player, Partial Controller Support, Family Sharing","Pixel Graphics, 2D Fighter, Action, Martial Arts, Swordplay, Combat, Nature, Rhythm, Spectacle fighter, Snow",Action_mid,0-5,"버그(1), 긍정 칭찬(1)"


'=== 2. 리뷰별 LLM 분석 결과 미리보기 ==='

,game_name,steam_label_text,llm_sentiment,sentiment_score,primary_issue,urgency,summary,suggested_action
0,Pirates VR: Jolly Roger,negative,negative,1,gameplay_loop,high,"아이템 찾기 플레이가 재미없고, 앵무새 캐릭터가 계속 플레이어를 모욕하여 매우 짜증난다는 부정적인 리뷰입니다.","어두운 곳에서 아이템을 찾는 플레이의 재미를 개선하기 위해 조명이나 힌트를 조절하는 것을 고려해야 합니다. 앵무새 캐릭터의 대사와 행동을 재평가하여 플레이어의 몰입을 방해하지 않도록 수정하거나, 음소거/등장 ..."
1,Pirates VR: Jolly Roger,negative,negative,1,gameplay_loop,high,"낮은 품질의 PCVR 게임이며, 새롭거나 독특한 점이 없고, 나쁜 게임 플레이와 형편없는 애니메이션 때문에 환불했다는 부정적인 리뷰입니다.",핵심 게임 플레이 메커니즘을 개선하여 VR 장르 내에서 더 매력적이고 독특하게 만들어야 합니다. 애니메이션 품질을 향상시켜 전반적인 시각적 완성도와 플레이어 몰입도를 높여야 합니다. 경쟁 게임 분석을 통해 게...
2,Pirates VR: Jolly Roger,positive,mixed,3,control,medium,"조작이 사용자 친화적이지 않지만, 게임의 아이디어와 구현은 좋다고 평가하는 긍정적인 리뷰입니다.",조작 체계를 검토하고 개선하여 사용자 친화성을 높여야 합니다. 사용자 정의 옵션이나 더 명확한 튜토리얼을 제공하는 것을 고려할 수 있습니다. 특정 조작 문제에 대한 피드백을 수집하여 개선할 부분을 정확히 파악...
3,Pirates VR: Jolly Roger,negative,negative,1,control,high,"플레이어는 수영 메커니즘, 앵무새, 클라이밍 시뮬레이션, 부족한 VR 상호작용(가방 상호작용 등)에 대해 불만을 표합니다.","수영 메커니즘을 개선하고, 앵무새의 방해 요소를 줄이거나 선택적으로 비활성화하는 옵션을 고려하며, VR 상호작용(예: 가방 상호작용)을 추가하여 게임 플레이의 몰입도를 높여야 합니다. 클라이밍 요소의 반복성을..."
4,Pirates VR: Jolly Roger,positive,mixed,3,content_volume,medium,"게임이 2017년 기준으로는 최고였겠지만 2025년에는 평범한 좋은 게임이며 특별한 놀라움은 없다고 평가합니다. 하지만 뛰어난 그래픽과 성우 연기, 재미있는 앵무새는 긍정적으로 언급합니다.","게임의 전반적인 혁신성과 현대적인 요소를 강화하여 2025년 기준에서도 '놀라움을 줄 수 있는' 게임이 되도록 콘텐츠 업데이트나 새로운 메커니즘 도입을 고려해야 합니다. 현재 긍정적인 평가를 받는 그래픽, 성..."
5,Pirates VR: Jolly Roger,positive,positive,5,positive_praise,low,게임이 재미있고 잘 실행된다는 점을 가장 중요하게 생각하며 긍정적으로 평가합니다.,"현재 게임의 안정성과 재미있는 요소를 유지하고, 플레이어들이 긍정적으로 평가하는 부분들을 지속적으로 강화하여 좋은 경험을 제공해야 합니다."
6,Pirates VR: Jolly Roger,positive,positive,5,positive_praise,low,"리뷰어는 VR과 해적 테마의 조합을 극찬하며, 게임이 자신을 위해 만들어진 것 같다고 표현했습니다. 데모를 플레이한 후 스포일러를 피하기 위해 바로 구매했으며, 아름다운 섬 배경과 앵무새 동반자와의 모험 시작...","현재의 게임 비전을 유지하고, 플레이어가 칭찬한 탐험, 상호작용적인 모험, 몰입감 있는 스토리 요소를 더욱 풍부하게 제공하는 데 집중하세요. 초기 경험의 높은 품질을 계속 유지하는 것이 중요합니다."
7,Pirates VR: Jolly Roger,negative,negative,1,gameplay_loop,high,"리뷰어는 게임이 상점 트레일러나 이미지에서 보여진 것과 전혀 다르다고 비판하며, 해적 게임으로 포장된 '방 탈출 및 등반 시뮬레이터'라고 묘사했습니다.","상점 페이지, 트레일러, 이미지를 검토하고 실제 게임 플레이(특히 '방 탈출' 및 '등반 시뮬레이터' 요소)를 정확하게 반영하도록 업데이트하세요. 게임의 핵심 메커니즘이 '해적 게임'에 대한 플레이어의 기대치..."
8,Pirates VR: Jolly Roger,positive,positive,4,positive_praise,low,"리뷰어는 게임 경험이 훌륭했다고 평가하며, 앵무새 동반자가 좋은 가이드 역할을 했다고 언급했습니다. 게임이 민첩성, 논리적 사고, 세부 사항에 대한 주의력을 요구한다고 강조했으며, 아름다운 자연 환경에 대해서...","앵무새와 같은 매력적인 동반자 캐릭터를 계속 개발하세요. 플레이어가 긍정적으로 평가한 민첩성, 논리적 사고, 세부 사항에 대한 주의력을 요구하는 게임 플레이 요소를 강조하고 확장하는 것을 고려하세요. 환경 디..."
9,Pirates VR: Jolly Roger,negative,negative,1,gameplay_loop,high,"플레이어는 게임의 비주얼은 훌륭하지만, 반복적인 암벽 등반과 어색한 수영 메커니즘, 멀미 옵션 부재 등으로 인해 게임 플레이가 지루하고 답답하다고 느꼈으며, 결국 환불을 요청했습니다.","핵심 게임 플레이(암벽 등반, 수영) 반복성 및 어색한 조작감 개선. VR 멀미 방지 옵션 추가. 수중 대사 오류 수정. 퀘스트 목표 명확성 개선."


'=== 3. 게임별 LLM 감정 분포 ==='

...

세부 이슈 태그,밸런스,버그,콘텐츠 분량,조작감,크래시,개발사 소통,난이도,게임플레이 루프,그래픽/사운드,과금,멀티/네트워크,최적화,기타,성능,긍정 칭찬,가격/가치,성장/반복 노가다,저장/진행,스토리,번역/현지화,UI/UX
게임명,,,,,,,,,,,,,,,,,,,,,
BOBR SPRINT,0,0,0,1,0,0,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0
Bachelairs,0,1,1,1,1,0,0,6,1,0,6,0,0,0,6,2,0,0,0,0,1
Bureau of Contacts,0,0,2,0,0,1,1,10,3,0,2,0,1,0,7,1,1,0,0,0,4
Dark Shadows,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,1,0,0
Detective Penguin,0,2,3,0,0,1,0,6,4,0,0,0,6,0,11,1,0,0,9,0,1
Dojo Masters,0,2,3,2,0,1,0,7,0,0,3,0,1,1,8,1,0,0,0,0,1
Echo Point Nova,0,1,0,0,0,1,2,11,2,1,1,1,0,0,6,1,1,0,0,0,2
First Cut: Samurai Duel,3,0,0,1,0,0,9,7,0,0,3,0,2,0,6,0,0,0,0,0,1
MoonWhite,0,1,1,0,0,0,0,2,3,0,0,0,0,0,1,0,0,0,0,0,0


'=== 8. 세부 이슈 태그별 감정 방향 분포 ==='

태그 감정,혼합,부정,중립,긍정
세부 이슈 태그,,,,
밸런스,0,6,0,1
버그,0,7,1,0
콘텐츠 분량,1,9,5,7
조작감,0,10,1,6
크래시,0,1,0,0
개발사 소통,0,2,1,3
난이도,0,16,4,7
게임플레이 루프,4,41,3,66
그래픽/사운드,1,6,1,51


'=== 9. 저장된 결과 파일 경로 확인 ==='

,결과 파일,경로
0,리뷰별 LLM 분석 결과 JSON,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\steam_review_llm_results.json
1,리뷰별 LLM 분석 결과 CSV,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\steam_review_llm_results.csv
2,게임 단위 요약 CSV,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\llm_game_summary.csv
3,세부 이슈 태그 CSV,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0_d30\llm_issue_tags_flat.csv


# 17. 프로젝트 주제 연결형 분석 결과 도출

1. 전체 LLM 분석 결과 요약
2. 세부 이슈별 긍정/부정 방향성
3. 게임별 운영 우선순위 점수
4. 게임별 강점과 개선 이슈
5. 출시 초기 구간별 주요 문제
6. 보고서에 바로 사용할 수 있는 핵심 인사이트

In [ ]:
# 17-1. 프로젝트 주제 연결형 분석 준비

df_insight = df_result.copy()
df_tags_insight = df_issue_tags_flat.copy()
game_summary_insight = game_summary.copy()

In [ ]:
# ============================================================
# 17-2. 전체 LLM 분석 결과 요약
# ============================================================
# 목적:
# - Steam 라벨만 볼 때와 LLM이 리뷰 본문을 읽고 판단한 감정이 어떻게 다른지 확인한다.
# - 단순 긍정률보다 '혼합/부정 의견이 얼마나 숨어 있는지'를 보는 것이 중요하다.

overall_summary = pd.DataFrame({
    "항목": [
        "분석 리뷰 수",
        "분석 게임 수",
        "Steam 긍정 리뷰 비율",
        "LLM 긍정 리뷰 비율",
        "LLM 부정+혼합 리뷰 비율",
        "긴급 개선 필요 리뷰 비율",
        "Steam 라벨-LLM 감정 완전 일치 비율",
    ],
    "값": [
        len(df_insight),
        df_insight["appid"].nunique(),
        f"{(df_insight['steam_label_text'].eq('positive').mean() * 100):.1f}%",
        f"{(df_insight['llm_sentiment'].eq('positive').mean() * 100):.1f}%",
        f"{(df_insight['llm_sentiment'].isin(['negative', 'mixed']).mean() * 100):.1f}%",
        f"{(df_insight['urgency'].eq('high').mean() * 100):.1f}%",
        f"{(df_insight['sentiment_relation'].eq('exact_match').mean() * 100):.1f}%" if "sentiment_relation" in df_insight.columns else "확인 불가",
    ]
})

display(overall_summary)

,항목,값
0,분석 리뷰 수,142
1,분석 게임 수,17
2,Steam 긍정 리뷰 비율,75.4%
3,LLM 긍정 리뷰 비율,60.6%
4,LLM 부정+혼합 리뷰 비율,37.3%
5,긴급 개선 필요 리뷰 비율,25.4%
6,Steam 라벨-LLM 감정 완전 일치 비율,81.7%


In [ ]:
# ============================================================
# 17-3. 세부 이슈별 긍정/부정 방향성 요약
# ============================================================
# 목적:
# - 어떤 이슈가 실제로 불만으로 많이 등장하는지 확인한다.
# - 단순 빈도뿐 아니라 부정 비율을 함께 보아야 패치 우선순위를 정할 수 있다.

issue_sentiment_summary = (
    df_tags_insight
    .groupby("tag_category")
    .agg(
        total_count=("tag_category", "count"),
        positive_count=("tag_sentiment", lambda x: (x == "positive").sum()),
        negative_count=("tag_sentiment", lambda x: (x == "negative").sum()),
        neutral_count=("tag_sentiment", lambda x: (x == "neutral").sum()),
    )
    .reset_index()
)

issue_sentiment_summary["negative_rate"] = (
    issue_sentiment_summary["negative_count"] / issue_sentiment_summary["total_count"]
)

issue_sentiment_summary["positive_rate"] = (
    issue_sentiment_summary["positive_count"] / issue_sentiment_summary["total_count"]
)

issue_sentiment_summary["issue_name_kor"] = (
    issue_sentiment_summary["tag_category"].map(ISSUE_KR_MAP).fillna(issue_sentiment_summary["tag_category"])
)

issue_sentiment_summary = issue_sentiment_summary.sort_values(
    ["negative_count", "negative_rate"],
    ascending=False
)

display(
    issue_sentiment_summary[
        ["issue_name_kor", "total_count", "negative_count", "negative_rate", "positive_count", "positive_rate"]
    ].head(15)
)

,issue_name_kor,total_count,negative_count,negative_rate,positive_count,positive_rate
7,게임플레이 루프,114,41,0.359649,66,0.578947
20,UI/UX,28,20,0.714286,8,0.285714
6,난이도,27,16,0.592593,7,0.259259
3,조작감,17,10,0.588235,6,0.352941
2,콘텐츠 분량,22,9,0.409091,7,0.318182
1,버그,8,7,0.875000,0,0.000000
12,기타,18,7,0.388889,8,0.444444
0,밸런스,7,6,0.857143,1,0.142857
8,그래픽/사운드,59,6,0.101695,51,0.864407
15,가격/가치,12,5,0.416667,6,0.500000


In [ ]:
# ============================================================
# 17-4. 게임별 운영 우선순위 점수 생성
# ============================================================
# 목적:
# - 게임별로 '지금 먼저 봐야 하는 게임'을 정한다.
# - high urgency는 2점, medium urgency는 1점으로 두고 0~1 사이의 위험 점수로 변환한다.
# - 리뷰 수가 5개 미만인 게임은 표본이 작으므로 별도 참고형으로 분리한다.

game_priority = game_summary_insight.copy()

game_priority["steam_positive_rate_sample"] = (
    game_priority["steam_positive_count"] / game_priority["analyzed_review_count"]
)

game_priority["high_urgency_rate"] = (
    game_priority["high_urgency_count"] / game_priority["analyzed_review_count"]
)

game_priority["medium_high_urgency_rate"] = (
    (game_priority["high_urgency_count"] + game_priority["medium_urgency_count"])
    / game_priority["analyzed_review_count"]
)

game_priority["operation_priority_score"] = (
    (game_priority["high_urgency_count"] * 2 + game_priority["medium_urgency_count"])
    / (game_priority["analyzed_review_count"] * 2)
)

def format_issue_count(series, top_n=3):
    counts = series.value_counts().head(top_n)
    if len(counts) == 0:
        return "해당 없음"
    return ", ".join([f"{ISSUE_KR_MAP.get(k, k)}({v})" for k, v in counts.items()])


negative_issue_rows = []
positive_issue_rows = []

for (appid, game_name), group in df_tags_insight.groupby(["appid", "game_name"]):
    negative_issue_rows.append({
        "appid": appid,
        "negative_issue_top3": format_issue_count(
            group.loc[group["tag_sentiment"] == "negative", "tag_category"],
            top_n=3
        )
    })

    positive_issue_rows.append({
        "appid": appid,
        "positive_strength_top3": format_issue_count(
            group.loc[group["tag_sentiment"] == "positive", "tag_category"],
            top_n=3
        )
    })

negative_issue_df = pd.DataFrame(negative_issue_rows)
positive_issue_df = pd.DataFrame(positive_issue_rows)

game_priority = (
    game_priority
    .merge(negative_issue_df, on="appid", how="left")
    .merge(positive_issue_df, on="appid", how="left")
)


def classify_operation_type(row):
    # 프로젝트 목적에 맞춘 운영 판단 유형 분류
    if row["analyzed_review_count"] < 5:
        return "소표본 참고형"

    if row["main_llm_sentiment"] == "negative" or row["high_urgency_rate"] >= 0.40:
        return "긴급 개선 필요형"

    if row["medium_high_urgency_rate"] >= 0.40:
        return "단기 개선 관리형"

    if row["main_llm_sentiment"] == "positive" and row["high_urgency_rate"] <= 0.20:
        return "강점 유지·확장형"

    return "모니터링형"


game_priority["operation_type"] = game_priority.apply(classify_operation_type, axis=1)

operation_priority_table = (
    game_priority
    .sort_values("operation_priority_score", ascending=False)
    [[
        "game_name",
        "analyzed_review_count",
        "steam_positive_rate_sample",
        "main_llm_sentiment",
        "high_urgency_rate",
        "medium_high_urgency_rate",
        "operation_priority_score",
        "operation_type",
        "negative_issue_top3",
        "positive_strength_top3",
    ]]
)

display(operation_priority_table)

,game_name,analyzed_review_count,steam_positive_rate_sample,main_llm_sentiment,high_urgency_rate,medium_high_urgency_rate,operation_priority_score,operation_type,negative_issue_top3,positive_strength_top3
10,My Life As An Alchemist,3,0.333333,negative,0.666667,1.000000,0.833333,소표본 참고형,게임플레이 루프(5),"긍정 칭찬(1), 가격/가치(1)"
0,Pirates VR: Jolly Roger,12,0.500000,negative,0.416667,0.750000,0.583333,긴급 개선 필요형,"게임플레이 루프(5), UI/UX(5), 조작감(5)","긍정 칭찬(6), 그래픽/사운드(4), 스토리(2)"
3,First Cut: Samurai Duel,12,0.500000,positive,0.500000,0.666667,0.583333,긴급 개선 필요형,"난이도(9), 게임플레이 루프(4), 밸런스(3)","긍정 칭찬(6), 게임플레이 루프(3), 멀티/네트워크(1)"
15,SLASHING NIGHT,1,1.000000,mixed,0.000000,1.000000,0.500000,소표본 참고형,해당 없음,가격/가치(1)
4,BOBR SPRINT,2,1.000000,mixed,0.500000,0.500000,0.500000,소표본 참고형,조작감(1),"긍정 칭찬(2), 그래픽/사운드(1)"
1,Echo Point Nova,12,0.500000,positive,0.416667,0.500000,0.458333,긴급 개선 필요형,"게임플레이 루프(8), UI/UX(1), 개발사 소통(1)","긍정 칭찬(6), 게임플레이 루프(3), 멀티/네트워크(1)"
11,Bureau of Contacts,12,0.666667,positive,0.333333,0.416667,0.375000,단기 개선 관리형,"게임플레이 루프(5), UI/UX(4), 콘텐츠 분량(2)","긍정 칭찬(7), 게임플레이 루프(4), 멀티/네트워크(2)"
5,Dojo Masters,12,0.750000,positive,0.250000,0.500000,0.375000,단기 개선 관리형,"버그(2), 멀티/네트워크(2), 성능(1)","긍정 칭찬(8), 게임플레이 루프(5), 조작감(1)"
12,Bachelairs,9,1.000000,positive,0.222222,0.444444,0.333333,단기 개선 관리형,"크래시(1), UI/UX(1), 버그(1)","긍정 칭찬(6), 멀티/네트워크(6), 게임플레이 루프(4)"
6,Zorro and Zedd,12,0.750000,positive,0.250000,0.333333,0.291667,모니터링형,"게임플레이 루프(6), UI/UX(2), 콘텐츠 분량(1)","긍정 칭찬(15), 게임플레이 루프(10), 그래픽/사운드(7)"


In [ ]:
INSIGHT_ISSUE_PRIORITY_PATH = OUTPUT_DIR / "llm_issue_priority_summary.csv"
INSIGHT_OPERATION_PRIORITY_PATH = OUTPUT_DIR / "llm_operation_priority.csv"

issue_sentiment_summary.to_csv(
    INSIGHT_ISSUE_PRIORITY_PATH,
    index=False,
    encoding="utf-8-sig"
)

operation_priority_table.to_csv(
    INSIGHT_OPERATION_PRIORITY_PATH,
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# 17-5. 출시 초기 구간별 이슈 변화 확인
# ============================================================
# 목적:
# - D0-D7과 D8-D30에서 어떤 이슈가 남는지 확인한다.
# - 출시 직후에는 조작/튜토리얼/초반 재미 문제가 중요하고,
#   시간이 지난 뒤에는 버그, 밸런스, 반복성 같은 문제가 남을 수 있다.

period_issue_table = pd.crosstab(
    df_insight["primary_issue"],
    df_insight["release_period"]
)

period_issue_table = period_issue_table.rename(index=ISSUE_KR_MAP)

display(period_issue_table.sort_values(by=list(period_issue_table.columns), ascending=False).head(15))

negative_period_tag_table = pd.crosstab(
    df_tags_insight.loc[df_tags_insight["tag_sentiment"] == "negative", "tag_category"],
    df_tags_insight.loc[df_tags_insight["tag_sentiment"] == "negative", "release_period"]
)

negative_period_tag_table = negative_period_tag_table.rename(index=ISSUE_KR_MAP)

display(negative_period_tag_table.sort_values(by=list(negative_period_tag_table.columns), ascending=False).head(15))

release_period,D0-D7,D8-D30
primary_issue,,
긍정 칭찬,57,23
게임플레이 루프,17,8
콘텐츠 분량,7,1
조작감,3,2
난이도,3,2
멀티/네트워크,3,1
UI/UX,3,0
버그,2,2
밸런스,2,1


release_period,D0-D7,D8-D30
tag_category,,
게임플레이 루프,27,14
UI/UX,11,9
난이도,7,9
조작감,6,4
콘텐츠 분량,6,3
밸런스,5,1
기타,4,3
그래픽/사운드,4,2
가격/가치,4,1


In [ ]:
# ============================================================
# 17-6. 보고서용 핵심 인사이트 후보 정리
# ============================================================
top_negative_issues = (
    issue_sentiment_summary
    .sort_values(["negative_count", "negative_rate"], ascending=False)
    .head(5)
)

urgent_games = (
    game_priority
    .sort_values("operation_priority_score", ascending=False)
    .head(5)
)

stable_games = (
    game_priority
    .query("operation_type == '강점 유지·확장형'")
    .sort_values(["steam_positive_rate_sample", "analyzed_review_count"], ascending=False)
    .head(5)
)

print("보고서용 후보 1. 주요 부정 이슈")
display(top_negative_issues)

print("보고서용 후보 2. 운영 우선순위가 높은 게임")
display(urgent_games)

print("보고서용 후보 3. 강점 유지·확장형 게임")
display(stable_games)

보고서용 후보 1. 주요 부정 이슈


,tag_category,total_count,positive_count,negative_count,neutral_count,negative_rate,positive_rate,issue_name_kor
7,gameplay_loop,114,66,41,3,0.359649,0.578947,게임플레이 루프
20,ui_ux,28,8,20,0,0.714286,0.285714,UI/UX
6,difficulty,27,7,16,4,0.592593,0.259259,난이도
3,control,17,6,10,1,0.588235,0.352941,조작감
2,content_volume,22,7,9,5,0.409091,0.318182,콘텐츠 분량


보고서용 후보 2. 운영 우선순위가 높은 게임


,appid,game_name,analyzed_review_count,steam_positive_count,steam_negative_count,main_llm_sentiment,main_issue,high_urgency_count,medium_urgency_count,low_urgency_count,release_date,genres,categories,top_tags,stratum,price_group,top_issues,steam_positive_rate_sample,high_urgency_rate,medium_high_urgency_rate,operation_priority_score,negative_issue_top3,positive_strength_top3,operation_type
10,2839500,My Life As An Alchemist,3,1,2,negative,gameplay_loop,2,1,0,2024-07-29,"Action, Adventure, Casual, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Family Sharing","Action Roguelike, Rogue-like, Pixel Graphics, Singleplayer, 2D, PvE, Casual, Tutorial, Arcade, Female Protagonist",Action_low,0-5,게임플레이 루프(3),0.333333,0.666667,1.000000,0.833333,게임플레이 루프(5),"긍정 칭찬(1), 가격/가치(1)",소표본 참고형
0,1443620,Pirates VR: Jolly Roger,12,6,6,negative,gameplay_loop,5,4,3,2025-01-14,"Action, Adventure, Indie","Single-player, Steam Achievements, Tracked Controller Support, VR Only, Family Sharing","Adventure, VR, Action, Pirates, Exploration, First-Person, Singleplayer, Action-Adventure, Choose Your Own Adventure...",Action_mid,10-20,"게임플레이 루프(4), 긍정 칭찬(3), 조작감(2)",0.500000,0.416667,0.750000,0.583333,"게임플레이 루프(5), UI/UX(5), 조작감(5)","긍정 칭찬(6), 그래픽/사운드(4), 스토리(2)",긴급 개선 필요형
3,2193490,First Cut: Samurai Duel,12,6,6,positive,positive_praise,6,2,4,2024-01-17,"Action, Indie","Single-player, Multi-player, PvP, Shared/Split Screen PvP, Shared/Split Screen, Steam Achievements, Full controller ...","Side Scroller, Hack and Slash, Martial Arts, 2D Fighter, PvP, PvE, Arcade, Swordplay, Character Customization, 2D",Action_high,5-10,"긍정 칭찬(4), 난이도(3), 멀티/네트워크(2)",0.500000,0.500000,0.666667,0.583333,"난이도(9), 게임플레이 루프(4), 밸런스(3)","긍정 칭찬(6), 게임플레이 루프(3), 멀티/네트워크(1)",긴급 개선 필요형
15,3291030,SLASHING NIGHT,1,1,0,mixed,content_volume,0,1,0,2024-11-14,"Action, Indie","Single-player, Family Sharing","Action, Female Protagonist, Bullet Time, Action-Adventure, Beat 'em up, Linear, Fantasy, Sci-fi, Combat, Indie",Action_low,0-5,콘텐츠 분량(1),1.000000,0.000000,1.000000,0.500000,해당 없음,가격/가치(1),소표본 참고형
4,2467000,BOBR SPRINT,2,2,0,mixed,control,1,0,1,2024-06-15,"Action, Adventure, Indie","Single-player, Full controller support, Family Sharing","2D Platformer, Adventure, Action-Adventure, Precision Platformer, Platformer, Pixel Graphics, Old School, 1990's, Du...",Action_mid,0-5,"조작감(1), 긍정 칭찬(1)",1.000000,0.500000,0.500000,0.500000,조작감(1),"긍정 칭찬(2), 그래픽/사운드(1)",소표본 참고형


보고서용 후보 3. 강점 유지·확장형 게임


,appid,game_name,analyzed_review_count,steam_positive_count,steam_negative_count,main_llm_sentiment,main_issue,high_urgency_count,medium_urgency_count,low_urgency_count,release_date,genres,categories,top_tags,stratum,price_group,top_issues,steam_positive_rate_sample,high_urgency_rate,medium_high_urgency_rate,operation_priority_score,negative_issue_top3,positive_strength_top3,operation_type
14,3191540,Smack Monkey,12,12,0,positive,positive_praise,0,1,11,2025-03-07,"Action, Adventure, Indie","Single-player, Family Sharing","Precision Platformer, Parkour, Pixel Graphics, Funny, Nature, Adventure, Cartoony, Platformer, Difficult, Puzzle-Pla...",Action_low,5-10,"긍정 칭찬(10), 게임플레이 루프(1), 난이도(1)",1.000000,0.000000,0.083333,0.041667,게임플레이 루프(1),"긍정 칭찬(10), 그래픽/사운드(6), 게임플레이 루프(5)",강점 유지·확장형
13,3110260,Detective Penguin,12,11,1,positive,positive_praise,1,1,10,2025-04-15,"Action, Adventure, Casual, Indie","Single-player, Family Sharing","Casual, Detective, Funny, Investigation, Cute, Cartoony, Conversation, Adventure, Interactive Fiction, Dark Humor",Action_low,5-10,"긍정 칭찬(10), 콘텐츠 분량(1), 게임플레이 루프(1)",0.916667,0.083333,0.166667,0.125000,"기타(2), 버그(1), 콘텐츠 분량(1)","긍정 칭찬(11), 스토리(9), 게임플레이 루프(5)",강점 유지·확장형
2,2008980,XENOTILT: HOSTILE PINBALL ACTION,12,10,2,positive,positive_praise,2,2,8,2024-11-12,"Action, Indie","Single-player, Steam Achievements, Full controller support, Steam Cloud, Stats, Steam Leaderboards, Family Sharing","Bullet Hell, Shoot 'Em Up, Difficult, Hack and Slash, Physics, Unforgiving, Arcade, 2D, Top-Down, Cyberpunk",Action_high,10-20,"긍정 칭찬(9), 기타(1), UI/UX(1)",0.833333,0.166667,0.333333,0.250000,"UI/UX(6), 기타(1), 콘텐츠 분량(1)","긍정 칭찬(9), 게임플레이 루프(6), 그래픽/사운드(5)",강점 유지·확장형
7,2738110,Vengeance Hunters,12,10,2,positive,positive_praise,2,2,8,2024-10-27,"Action, Indie","Single-player, Multi-player, Co-op, Shared/Split Screen Co-op, Shared/Split Screen, Steam Achievements, Full control...","Action, Beat 'em up, Pixel Graphics, Post-apocalyptic, Retro, Combat, Indie, Local Co-Op, Multiplayer, Arcade",Action_mid,10-20,"긍정 칭찬(8), 게임플레이 루프(2), 밸런스(1)",0.833333,0.166667,0.333333,0.250000,"게임플레이 루프(4), 난이도(4), 밸런스(3)","게임플레이 루프(13), 그래픽/사운드(13), 긍정 칭찬(10)",강점 유지·확장형


# 18. 사용 메모

## 실제 실행 순서

1. `ROOT` 경로를 본인 프로젝트 경로에 맞게 확인한다.
2. `.env`에 `GEMINI_API_KEY`가 있는지 확인한다.
3. 처음에는 `RUN_LLM=False`, `RUN_CHECK_CELLS=True`로 데이터 구조만 확인한다.
4. 필터 조건을 조절한다.
5. `TEST_N=30` 정도로 두고 `RUN_LLM=True`로 테스트한다.
6. 결과가 괜찮으면 `TEST_N=None` 또는 더 큰 값으로 확대한다.


## 확인용 코드 삭제 기준
- `확인용`이라고 적힌 셀은 출력 점검용이다.
- 해당 셀을 삭제해도 데이터 로드, 필터링, 샘플링, LLM 호출, 결과 저장 흐름은 유지된다.
- 단, 제목이 `데이터 로드`, `전처리`, `분석 실행`인 핵심 코드 셀은 삭제하면 안 된다.

## 이 노트북의 결과 파일

| 파일 | 내용 |
|---|---|
| `llm_review_analysis_result.json` | 리뷰별 LLM 분석 결과 JSON |
| `llm_review_analysis_result.csv` | 리뷰별 LLM 분석 결과 CSV |
| `llm_review_analysis_checkpoint.json` | 중간 저장 파일 |
| `llm_game_summary.csv` | 게임 단위 요약표 |
| `llm_issue_tags_flat.csv` | 이슈 태그 펼친 결과 |

# 확인용

In [ ]:
df_result["analysis_status"].value_counts(dropna=False)

analysis_status
success    142
Name: count, dtype: int64

In [ ]:
df_result[df_result["analysis_status"] != "success"].head(10)

,analysis_status,recommendationid,appid,game_name,language,review_datetime,timestamp_created,steam_voted_up,steam_label_text,playtime_at_review_hours,playtime_forever_hours,playtime_stage,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,release_date,days_from_release,release_period,genres_text,categories_text,top_steam_tags_text,price,price_group,stratum,llm_sentiment,sentiment_score,primary_issue,issue_tags,urgency,summary,suggested_action,sentiment_relation


In [ ]:
df_result["analysis_status"].value_counts(dropna=False)

analysis_status
success    142
Name: count, dtype: int64

In [ ]:
df_result["sentiment_relation"].value_counts(dropna=False)

sentiment_relation
exact_match      116
partial_match     23
unclear            3
Name: count, dtype: int64

In [ ]:
df_result.groupby(["appid", "game_name"])["recommendationid"].count().sort_values()

appid    game_name                       
3291030  SLASHING NIGHT                       1
3375300  Dark Shadows                         1
2812480  MoonWhite                            2
2467000  BOBR SPRINT                          2
2839500  My Life As An Alchemist              3
2790180  The Recurrence                       4
2948380  Bachelairs                           9
2008980  XENOTILT: HOSTILE PINBALL ACTION    12
2738110  Vengeance Hunters                   12
2599630  Zorro and Zedd                      12
2538000  Dojo Masters                        12
2193490  First Cut: Samurai Duel             12
1836730  Echo Point Nova                     12
2840210  Bureau of Contacts                  12
3110260  Detective Penguin                   12
1443620  Pirates VR: Jolly Roger             12
3191540  Smack Monkey                        12
Name: recommendationid, dtype: int64